**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/moaddab-2021-ventral"], check=True)
    os.chdir("/content/encore/entries/moaddab-2021-ventral")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Moaddab, Ray & McDannald (2021) from CRCNS data set bf-3

**Paper.** Moaddab M, Ray MH, McDannald MA (2021). Ventral pallidum neurons dynamically signal
relative threat. *Communications Biology* 4, 43.
[doi:10.1038/s42003-020-01554-4](https://doi.org/10.1038/s42003-020-01554-4). The claims file
(`claims/crcns-bf-3/10.1038-s42003-020-01554-4.yaml`) was written from the version of record
(PMC7794503) without its Supplementary Information.

**Data set.** CRCNS bf-3, "Recordings from ventral pallidum neurons in male rats undergoing fear
discrimination" (Moaddab and McDannald),
[doi:10.6080/K0HH6H8V](https://doi.org/10.6080/K0HH6H8V), read from the public bucket at
`https://crcnsarchive.s3-us-west-2.amazonaws.com/bf-3/live/`. CRCNS data sets are not versioned, so
the version is recorded as `unversioned` and pinned by checksum: the one data file read,
`data_and_scripts.zip`, has MD5 `43a2887fac54e7bd12db345257a29864` in the data set's
`checksums.md5`, and the notebook checks the MD5 of the bytes it reads against it. This is the only
copy the archive serves (the `live/` folder; there is no `old/` folder), and its data descriptor is
version 0.6 of 3 February 2021, the initial version, so there was no other version to choose.

**What the files hold.** One MATLAB file per unit (435) with spike times, a mean waveform, and the
session's Med Associates events as Plexon channels; a cube of all units precomputed by the
authors; and the authors' MATLAB scripts `cubeScriptPR25.m` and `vpAnalysis.m`. The paper does not
link these scripts (it points to a lab page that no longer resolves), but they are the authors' and
are used here, as the protocol uses a paper's code: where the Methods leave a detail open, the
scripts settle it; where they disagree with the Methods, the Methods' reading is the main result
and the scripts' is shown as a variant. The one exception is the neuron groups (A1). The cells
after the data check print the script lines cited here. Every number is computed from the spike and event times;
the cube is used only for checks.

**Assumptions the outcomes rest on** (marked A1 to A3 in the table):

- A1. The neuron groups are the paper's 257 cue-responsive neurons and their split into 131
  cue-excited and 126 cue-inhibited neurons, as the authors' code computes them: rates floored to
  integers in the screen (the Methods say "mean firing rate (Hz)" and do not mention rounding), and
  the code's per-trial normalization for the split. Both steps depart from the rule above. They are
  taken from the code because the claims are about the neurons the paper analysed, and the code
  reproduces every count the paper reports (257 neurons from 153 sessions, 131 and 126). The
  Methods' unfloored rates give 271 neurons, whose split is outside tolerance under every reading
  of the claims file (groups B, shown as a variant where it bears on a claim). On the 257 neurons,
  the claims-file reading Hz with two window means (130/127), one of the readings within tolerance,
  puts 25 neurons in the other group (claim fig2-excited-inhibited-split).
- A2. The cue-inhibited clusters are those of one Matlab k-means++ start after `rng(1)`, as the
  authors' code runs it (Low 74, Intermediate 34, High 18). The claims file's reading (b) for the
  ISIs with raw features and the same start gives the same members. With more restarts k-means
  finds other clusters with a lower sum of squares, so the clusters depend on the start.
- A3. A nose-poke pause is a gap of more than 2 s between pokes 18 to 2 s before a cue onset, as
  the authors' code defines it; the text read does not define it.

Firing measures use the Methods' normalization under the claims file's readings; the authors'
per-trial normalization, which reproduces the paper's printed values, is a variant throughout.
Changing one element at a time shows that Z-scoring single trials in place of trial means accounts
for the difference (section "Which part of the normalization", before claim fig3b).

| Claim | Tested | Rests on |
|---|---|---|
| recorded-units | tested |  |
| cue-responsive-screen | tested |  |
| fig1f-suppression-cue-effect | tested |  |
| fig1f-differential-suppression | tested |  |
| fig2-excited-inhibited-split | tested | A1 |
| ancova-excited-no-baseline-effect | tested | A1 |
| ancova-inhibited-baseline-effect | tested |  |
| kmeans-three-clusters | tested | A2 |
| clusters-differ-in-cue-firing | tested | A2 |
| clusters-by-rat | tested | A2 |
| fig3a-low-differential-cue-firing | tested | A2 |
| fig3b-low-inhibited-to-threat-cues | tested | A2 |
| fig3b-low-differential | tested | A2 |
| fig3c-intermediate-differential-cue-firing | tested | A2 |
| fig3d-intermediate-inhibition | tested | A2 |
| fig3d-intermediate-differential | tested | A2 |
| supp5-low-cue-correlations | tested | A2 |
| supp6-low-nosepoke-pauses | tested | A2, A3 |
| fig4a-low-rt-over-fo | tested | A2 |
| fig4a-low-025-better-than-000 | tested | A2 |
| fig4a-low-025-similar-050 | tested | A2 |
| fig4b-low-dynamic-shift | tested | A2 |
| fig4c-low-early-late-betas | tested | A2 |
| fig4d-low-early-late-shift | tested | A2 |
| fig4e-int-025-better-than-000 | tested | A2 |
| fig4e-int-025-similar-050 | tested | A2 |
| fig4f-int-no-assignment-interval | tested | A2 |
| fig4g-int-early-late-betas | tested | A2 |
| fig4h-int-early-late-shift | tested | A2 |
| fig4-low-int-combined | tested | A2 |
| fig5a-low-reward-firing | tested | A2 |
| fig5bc-reward-pre-post | tested | A2 |
| fig5d-f-low-reward-cue-correlations | tested | A2 |
| fig5-low-fisher-comparisons | tested | A2 |
| fig5g-i-intermediate-no-reward-cue-correlation | tested | A2 |
| fig6a-excited-differential-cue-firing | tested | A1 |
| fig6b-excited-period-firing | tested | A1 |
| supp8-excited-cue-correlations | tested | A1 |
| supp9-excited-nosepoke-pauses | tested | A1, A3 |
| fig6-excited-assignment-050 | tested | A1 |
| fig6c-excited-regressor-interval | tested | A1 |
| fig6d-excited-betas | tested | A1 |
| fig6ef-excited-reward-firing | tested | A1 |
| fig6g-excited-reward-cue-correlations | tested | A1 |
| fig6-excited-peak-cue-vs-reward | tested | A1 |

All 45 claims are tested; none is blocked.

In [ ]:
import time
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import vp

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
zf, md5 = vp.read_zip()
units = vp.load_units(zf)
cube = vp.load_cube(zf)
for u in units:
    vp.unit_measures(u)
print(f"read {vp.BASE_URL}{vp.ZIP_NAME}: MD5 {md5}, checksums.md5 gives {vp.ZIP_MD5}, "
      f"match {md5 == vp.ZIP_MD5}")
print(f"{len(units)} unit files loaded in {time.time() - started:.1f} s")


def seed_for(claim_id: str) -> np.random.Generator:
    """Each claim section has its own generator, seeded from its id (a standard choice)."""
    return np.random.default_rng(sum(ord(c) * (i + 1) for i, c in enumerate(claim_id)))


def ci_text(m, lo, hi):
    return f"M = {m:.3f}, 95% CI [{lo:.3f}, {hi:.3f}]"

## Data check

In [ ]:
info = pd.DataFrame([{"rat": u["rat"], "session": u["session"], "program": u["program"],
                      "trials": len(u["onsets"]), "spikes": len(u["spikes"]),
                      "duration_s": u["spikes"][-1] - u["spikes"][0]} for u in units])
print(f"units {len(info)}, rats {info.rat.nunique()}, sessions (distinct file names) {info.session.nunique()}")
print("rat and date pairs with more than one session name:",
      sum(g.session.nunique() > 1 for _, g in info.assign(date=[u["date"] for u in units]).groupby(["rat", "date"])))
print("\nunits and sessions per rat:")
print(info.groupby("rat").agg(units=("session", "size"), sessions=("session", "nunique")).T.to_string())
print("\nunits by behavior program letter:", dict(Counter(info.program)))
print("event channels present (set: units):", dict(Counter(tuple(sorted(u["events"])) for u in units)))
print("\ntrials per unit:", dict(Counter(info.trials)))
print("trial types per unit (danger, unc. shock, unc. omission, safety):",
      dict(Counter(tuple(int(x) for x in np.bincount(u["types"], minlength=5)[1:]) for u in units)))
print("spike times sorted in every file:", all(u["spikes_sorted_in_file"] for u in units))
rates = np.array([len(u["spikes"]) / (u["spikes"][-1] - u["spikes"][0]) for u in units])
print(f"session-mean firing rate: {rates.min():.2f} to {rates.max():.2f} Hz (median {np.median(rates):.2f})")
print(f"recording span from first to last spike: {info.duration_s.min() / 60:.1f} to {info.duration_s.max() / 60:.1f} min")
ws = np.array([u["wave"] for u in units])
print(f"waveforms: {ws.shape[1]} samples each; values {ws.min():.4f} to {ws.max():.4f} (units not stated)")

The data descriptor does not name the event channels. The authors' script `cubeScriptPR25.m`
gives them (EVT01 shock, EVT02 pellet, EVT13 nose poke, cue channels by program letter;
`vp.CUE_CHANNELS`). The check below tests the cue mapping against the shocks: danger trials must
all be followed by a shock 12 s after cue onset (10 s cue, 2 s delay), uncertainty trials in 2 of 8,
safety never.

In [ ]:
rows = []
for s, g in info.groupby("session"):
    u = units[g.index[0]]
    shock = u["events"].get("EVT01", np.array([]))
    for t, ty in zip(u["onsets"], u["types"]):
        lag = shock[(shock > t) & (shock <= t + 30)] - t
        rows.append({"session": s, "type": vp.TYPE_NAMES[ty], "shocked": len(lag) > 0,
                     "lag": lag[0] if len(lag) else np.nan})
sh = pd.DataFrame(rows)
print(sh.groupby("type").agg(trials=("shocked", "size"), shocked=("shocked", "sum"),
                             lag_min=("lag", "min"), lag_max=("lag", "max")).to_string())
print("sessions:", sh.session.nunique(), "; shocks outside a trial window:",
      sum(len(units[g.index[0]]["events"]["EVT01"]) for _, g in info.groupby("session")) - int(sh.shocked.sum()))
print("EVT08 equals EVT01 in every unit:", all(np.array_equal(u["events"]["EVT01"], u["events"]["EVT08"]) for u in units))
gaps = np.concatenate([np.diff(units[g.index[0]]["onsets"]) for _, g in info.groupby("session")])
print(f"cue onset to next cue onset: {gaps.min():.0f} to {gaps.max():.0f} s, mean {gaps.mean():.0f} s "
      f"(paper: mean inter-trial interval 3.5 min)")

The trial tables built here are compared with the authors' cube (`cube.tag`, trial type and cue
onset per unit), and the cube's floored 10 s baseline rates and ISI coefficients with values
computed here from the spike times.

In [ ]:
cube_names = ["".join(cube.name[k][i] for k in range(22)) for i in range(cube.tag.shape[2])]
cidx = {n: i for i, n in enumerate(cube_names)}
same_tag = same_base = 0
cv_diff = []
for u in units:
    i = cidx[u["name"]]
    t = cube.tag[:, :, i]
    o = np.argsort(t[:, 1])
    same_tag += np.allclose(t[o, 1], u["onsets"]) and np.array_equal(t[o, 0].astype(int), u["types"])
    same_base += np.array_equal(cube.fire.raw.base10[o, 0, i] if cube.fire.raw.base10.ndim == 3 else cube.fire.raw.base10[o, i],
                                np.floor(u["base10"]))
    cv_diff.append(abs(cube.fire.CV[i] - vp.isi_stats(u, "base20")[0]))
print(f"units in the cube: {len(cube_names)}; unit files found in the cube by name: {sum(u['name'] in cidx for u in units)}")
print(f"trial times and types equal to the cube's: {same_tag} of {len(units)}")
print(f"cube 10 s baseline rates equal to floor(count / 10): {same_base} of {len(units)}")
print(f"cube CV against CV of ISIs in the 20 s before each cue: largest difference {max(cv_diff):.2e}")
print("cube rows in type order (rows 1-4 danger, 5-6 unc. shock, 7-12 unc. omission, 13-16 safety, "
      "as the descriptor says):", all(np.array_equal(cube.tag[:, 0, i], [1] * 4 + [2] * 2 + [3] * 6 + [4] * 4)
                                       for i in range(len(cube_names))))
pb = np.array([np.allclose(np.sort(cube.poke.base[:, i]) if cube.poke.base.ndim == 2 else 0,
                           np.sort(units[k]["poke_base"])) for k, i in [(k, cidx[u["name"]]) for k, u in enumerate(units)]])
print(f"cube.poke.base equal to the poke rate over the 20 s before cue onset (pokes/min): {pb.sum()} of {len(units)}; "
      "the descriptor says 2 s")
print("cube fields:", cube._fieldnames, "; waveform characteristics stored in the cube:",
      any(f in cube._fieldnames for f in ("halfDuration", "amplitudeRatio")))
print(f"stored waveforms per unit: one, of {units[0]['wave'].size} samples (descriptor: 'mean waveform values for each trial type')")

Units recorded more than once. Pairs of units from the same session whose spike times coincide
(within 0.1 ms) for more than half of the spikes of the first:

In [ ]:
dup_rows = []
for s, g in info.groupby("session"):
    us = [units[i] for i in g.index]
    for i in range(len(us)):
        for j in range(i + 1, len(us)):
            a, b = us[i]["spikes"], us[j]["spikes"]
            k = np.clip(np.searchsorted(b, a), 1, len(b) - 1)
            frac = (np.minimum(abs(b[k] - a), abs(b[k - 1] - a)) < 1e-4).mean()
            if frac > 0.1:
                dup_rows.append({"unit 1": us[i]["name"], "unit 2": us[j]["name"], "spikes 1": len(a),
                                 "spikes 2": len(b), "fraction of unit 1 spikes shared": round(frac, 3)})
dups = pd.DataFrame(dup_rows)
print(dups.to_string(index=False))
print("pairs of units with identical spike trains:", sum(np.array_equal(units[i]["spikes"], units[j]["spikes"])
      for i in range(len(units)) for j in range(i + 1, len(units)) if len(units[i]["spikes"]) == len(units[j]["spikes"])))

Recorded time. The gaps in every spike train suggest that recording was paused between trials.
The check below finds, for every trial, the first spike after the preceding gap of more than
30 s, and the EVT14 events ("pause recording" in `cubeScriptPR25.m`), relative to cue onset, and
counts the pokes and pellets that fall within 20 s before to 31.5 s after a cue onset.

In [ ]:
first_spike, ev14, pk_in, pl_in, pk_all, pl_all = [], [], 0, 0, 0, 0
for u in units:
    sp, on = u["spikes"], u["onsets"]
    g = np.where(np.diff(sp) > 30)[0]
    starts = np.r_[sp[0], sp[g + 1]]
    for t in on:
        before = starts[starts <= t]
        if len(before):
            first_spike.append(before.max() - t)
    e = u["events"]["EVT14"]
    ev14 += [e[e > t].min() - t for t in on if np.any(e > t)]
win_check = pd.Series({"first spike after a gap, median s from onset": np.median(first_spike),
                       "same, 5th percentile": np.percentile(first_spike, 5),
                       "EVT14 after onset, min s": np.min(ev14), "EVT14 after onset, max s": np.max(ev14),
                       "EVT14 events per unit": np.mean([len(u["events"]["EVT14"]) for u in units])})
print(win_check.round(2).to_string())
for name, ch in (("nose pokes (EVT13)", "EVT13"), ("pellets (EVT02)", "EVT02")):
    n_in = sum(vp.inside(u["events"][ch], vp.recorded_windows(u), 0, 0).sum() for u in units)
    n_all = sum(len(u["events"][ch]) for u in units)
    print(f"{name}: {n_in} of {n_all} within 20 s before to the EVT14 after a cue onset")

Clock check. Spikes and events come from one Plexon file per session. If they share a clock, firing
and nose poking should tell danger from safety trials only after cue onset. For each 1 s bin from
10 s before to 12 s after cue onset: the fraction of units whose firing differs between the 4 danger
and 4 safety trials (Welch t-test, p < 0.05; chance 0.05), and the mean poke rate on danger and
safety trials.

In [ ]:
edges = np.arange(-10, 13)
clock = []
for k in range(len(edges) - 1):
    sig = []
    for u in units:
        c = vp.counts(u["spikes"], u["onsets"] + edges[k], u["onsets"] + edges[k + 1])
        a, b = c[u["types"] == vp.DANGER], c[u["types"] == vp.SAFETY]
        sig.append(stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05 if np.std(np.r_[a, b]) > 0 else False)
    pk = [vp.counts(u["events"]["EVT13"], u["onsets"] + edges[k], u["onsets"] + edges[k + 1]) * 60 for u in units]
    pk_d = np.mean([p[u["types"] == vp.DANGER].mean() for p, u in zip(pk, units)])
    pk_s = np.mean([p[u["types"] == vp.SAFETY].mean() for p, u in zip(pk, units)])
    clock.append({"bin_s": f"{edges[k]} to {edges[k + 1]}", "units danger != safety": np.mean(sig),
                  "pokes/min danger": pk_d, "pokes/min safety": pk_s})
print(pd.DataFrame(clock).round(3).to_string(index=False))

The authors' scripts. The lines cited in this notebook and in `vp.py`, printed from the copies
in the data set's zip, so that each detail taken from the code can be checked against it.

In [ ]:
scripts = {n.split("/")[-1]: zf.read(n).decode("latin-1").splitlines() for n in zf.namelist() if n.endswith(".m")}
CITED = [("cubeScriptPR25.m", 143, 163, "pellet deliveries 18 to 2 s before and 14 to 28 s after each cue onset"),
         ("cubeScriptPR25.m", 170, 193, "nose-poke pauses: gaps over 2 s between pokes 18 to 2 s before a cue onset"),
         ("cubeScriptPR25.m", 262, 295, "2 s, 5 s and 10 s baseline rates per trial, floored"),
         ("cubeScriptPR25.m", 329, 338, "cue onset rate per trial, floored"),
         ("cubeScriptPR25.m", 373, 382, "last 5 s rate per trial, floored"),
         ("cubeScriptPR25.m", 456, 472, "per-trial firing minus the trial's 2 s baseline, Z over all trials and bins"),
         ("cubeScriptPR25.m", 556, 569, "pellet firing minus the first 1 s of the window, Z over all events and bins"),
         ("cubeScriptPR25.m", 670, 684, "pause firing, the same"),
         ("vpAnalysis.m", 7, 64, "cue screen: ttest of floored window rates against the floored 10 s baseline"),
         ("vpAnalysis.m", 1182, 1205, "split: danger late-cue mean plus onset mean of the per-trial Z, sign"),
         ("vpAnalysis.m", 1485, 1487, "k-means: rng(1), raw features, Matlab defaults"),
         ("vpAnalysis.m", 1719, 1727, "Low cluster cue correlations over the first 5 s"),
         ("vpAnalysis.m", 2776, 2779, "cue-excited: first cue bin and first reward bin minus the bin before"),
         ("vpAnalysis.m", 3902, 3904, "Low: second cue bin minus the bin before cue onset")]
for f, a, b, what in CITED:
    print(f"==== {f}, lines {a}-{b}: {what}")
    print("\n".join(f"{k:5d}  {scripts[f][k - 1]}" for k in range(a, b + 1) if scripts[f][k - 1].strip()))


## Claim recorded-units

> 435 VP neurons were recorded from 14 rats over 194 sessions.

In [ ]:
n_units, n_rats, n_sessions = len(units), info.rat.nunique(), info.session.nunique()
print(f"neurons {n_units} (expected 435), rats {n_rats} (expected 14), sessions {n_sessions} (expected 194)")
print("sessions are named by the session file name; no rat has two session names on one date, and no two "
      "sessions share a trial table (checked below)")
tt = Counter(tuple(np.round(units[g.index[0]]["onsets"], 3)) for _, g in info.groupby("session"))
print("distinct trial tables:", len(tt), "of", n_sessions, "sessions")
print(f"same-session unit pairs sharing more than 70% of spikes: {(dups['fraction of unit 1 spikes shared'] > 0.7).sum()}")

**Result: partially reproduced.** The neuron and rat counts match (435, 14); the files hold 195
sessions, one more than the paper's 194. Each of the 195 session names has its own trial table. Four pairs of units from
one session share 70 to 88% of their spikes and may be one neuron recorded on two wires; the paper
counts them as separate neurons, and so do the analyses below.

## Claim cue-responsive-screen

> 257 of the 435 neurons (about 59%) were cue-responsive, recorded in 153 sessions, with at least
> one cue-responsive neuron in each of the 14 rats.

The criterion follows the Methods: mean rate (Hz) in the first 1 s and the last 5 s of each cue
against the 10 s baseline, paired two-tailed t-tests, p < 0.05, no correction. The authors' code
(`vpAnalysis.m`, lines 7-64, on the cube built by `cubeScriptPR25.m`) floors each trial's rate to
an integer before the tests; it is shown as a variant.

In [ ]:
screen_p = {"methods": np.array([vp.screen(u) for u in units]),
            "code (floored rates)": np.array([vp.screen(u, floor_rates=True) for u in units])}
POP = {}
rows = []
for k, p in screen_p.items():
    m = np.nanmin(p, 1) < 0.05
    POP[k] = m
    rats_with = {u["rat"] for u, x in zip(units, m) if x}
    rows.append({"rates": k, "cue-responsive": int(m.sum()), "fraction": m.mean(),
                 "sessions": len({u["session"] for u, x in zip(units, m) if x}), "rats with one": len(rats_with),
                 "within tolerance (257 +- 25.7, 153 +- 15.3)": abs(m.sum() - 257) <= 25.7
                 and abs(len({u['session'] for u, x in zip(units, m) if x}) - 153) <= 15.3 and len(rats_with) == 14})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nunits passing each of the six tests (methods rates): ",
      dict(zip(["D onset", "U onset", "S onset", "D last5", "U last5", "S last5"], (screen_p["methods"] < 0.05).sum(0))))
both = POP["methods"] & POP["code (floored rates)"]
print(f"units responsive under both: {both.sum()}; methods only {(POP['methods'] & ~both).sum()}; "
      f"code only {(POP['code (floored rates)'] & ~both).sum()}")
by_rat = pd.DataFrame({"rat": info.rat, "methods": POP["methods"], "code": POP["code (floored rates)"]}).groupby("rat").sum()
print("\ncue-responsive units per rat:\n" + by_rat.T.to_string())

# chance reference (the paper states none): six uncorrected tests at 0.05 pass a neuron with no
# cue response with probability up to 1 - 0.95**6 if independent. Empirical: the same screen with
# no cue, the 10 s from 20 to 10 s before onset as "baseline" and the windows 10 to 9 s and 5 to 0 s
# before onset as "onset" and "last 5 s".
def pseudo_screen(u):
    sp, on = u["spikes"], u["onsets"]
    base, w1, w5 = (vp.counts(sp, on + a, on + b) / (b - a) for a, b in ((-20, -10), (-10, -9), (-5, 0)))
    p = []
    for win in (w1, w5):
        for sel in (u["types"] == vp.DANGER, np.isin(u["types"], [vp.UNC_SHOCK, vp.UNC_OMIT]), u["types"] == vp.SAFETY):
            d = win[sel] - base[sel]
            p.append((0.0 if d[0] != 0 else np.nan) if np.all(d == d[0]) else stats.ttest_rel(win[sel], base[sel]).pvalue)
    return np.nanmin(p) < 0.05
pseudo = np.array([pseudo_screen(u) for u in units])
print(f"\nchance: 1 - 0.95**6 = {1 - 0.95 ** 6:.3f} ({(1 - 0.95 ** 6) * 435:.0f} of 435) for independent tests; "
      f"the same screen on pre-cue windows passes {pseudo.sum()} of 435 ({pseudo.mean():.3f}), against 271 (0.623) on the cue")


**Result: reproduced.** With the Methods' rates, 271 of 435 neurons (62%) from 160 sessions are
cue-responsive, within the tolerance of 257 and 153, and every rat has one. With the rates floored as
in the authors' code, the screen gives 257 neurons from 153 sessions, the paper's counts exactly. The paper gives no chance level; the same screen on windows before the cue passes 119 of 435 neurons (27%, near the 26.5% that six independent tests at 0.05 give), against 271 (62%) on the cue. 119 is 44% of 271, an upper bound on the share of the cue-responsive neurons expected to pass by chance.

The paper's later analyses use these 257 neurons. Below, unless a section says otherwise, the
neuron groups are the ones the authors' code produces (assumption A1, stated in the summary): the
257 neurons of the floored screen, split into cue-excited and cue-inhibited neurons, and the
cue-inhibited neurons clustered, as `vpAnalysis.m` does. The groups the Methods' own rates give
are shown as a variant where they bear on a claim.

## Claim fig1f-suppression-cue-effect

> Suppression ratios were high to danger, intermediate to uncertainty and low to safety, with a main
> effect of cue in an ANOVA on the rats' mean suppression ratios.

Per trial, (baseline - cue)/(baseline + cue) poke rate, baseline the 20 s before cue onset, cue the
10 s cue (Methods). Per session and cue the mean over trials, then per rat the mean over the
sessions with cue-responsive neurons, each session counted (a) once or (b) once per cue-responsive
neuron; trials with no pokes in baseline or cue are (a) dropped or (b) set to 0 (claims file, O3).
The authors' code sets them to 1 and counts each session once (`vpAnalysis.m`, lines 431-434 and
1910-1933), shown as a variant.

In [ ]:
def sr_by_rat(pop_mask, nan_rule, weight):
    rows = []
    for u, m in zip(units, pop_mask):
        if not m:
            continue
        sr = vp.fear_output(u, nan_rule)
        rows.append({"rat": u["rat"], "session": u["session"],
                     **{c: np.nanmean(sr[sel]) for c, sel in (("danger", u["types"] == vp.DANGER),
                        ("uncertainty", np.isin(u["types"], [vp.UNC_SHOCK, vp.UNC_OMIT])), ("safety", u["types"] == vp.SAFETY))}})
    df = pd.DataFrame(rows)
    if weight == "session":
        df = df.drop_duplicates("session")
    return df.groupby("rat")[["danger", "uncertainty", "safety"]].mean()

n_00 = sum(np.isnan(u["sr"]).sum() for u, m in zip(units, POP["code (floored rates)"]) if m)
n_00_s = sum(np.isnan(units[g.index[0]]["sr"]).sum() for _, g in info[POP["code (floored rates)"]].groupby("session"))
print(f"0/0 trials: {n_00_s} of {16 * info[POP['code (floored rates)']].session.nunique()} trials in the "
      f"sessions with cue-responsive neurons")
SR = {}
rows = []
for pop in ("code (floored rates)", "methods"):
    for nan_rule in ("drop", "zero", "one"):
        for weight in ("session", "neuron"):
            r = sr_by_rat(POP[pop], nan_rule, weight)
            SR[(pop, nan_rule, weight)] = r
            a = vp.rm_anova(r.values, ["cue"])["cue"]
            rows.append({"population": pop, "0/0 trials": nan_rule, "per rat over": weight, "rats": len(r),
                         "danger": r.danger.mean(), "uncertainty": r.uncertainty.mean(), "safety": r.safety.mean(),
                         "F": a[0], "df": f"{a[1]},{a[2]}", "p": f"{a[3]:.3g}", "eta2p": a[4],
                         "criterion met": a[3] < 0.05 and r.danger.mean() > r.uncertainty.mean() > r.safety.mean()})
fig1f = pd.DataFrame(rows)
print(fig1f.round(4).to_string(index=False))
print("\nrat means, main population, 0/0 dropped, sessions counted once:")
print(SR[("code (floored rates)", "drop", "session")].round(3).T.to_string())

**Result: reproduced.** Under every reading the rat means order danger (0.92 to 0.93) > uncertainty
(0.62 to 0.64) > safety (0.11 to 0.13), and the one-way repeated measures ANOVA on the 14 rats gives
a cue effect (main reading F(2,26) = 75.80, partial eta squared 0.85; paper F(2,26) = 75.34, 0.85).

## Claim fig1f-differential-suppression

> Suppression ratios differed for every cue pair; the 95% bootstrap intervals of the differences
> excluded zero for danger vs. uncertainty (0.28), uncertainty vs. safety (0.52) and danger vs.
> safety (0.80).

In [ ]:
rng = seed_for("fig1f-differential-suppression")
expected = {"danger - uncertainty": (0.28, 0.10), "uncertainty - safety": (0.52, 0.17), "danger - safety": (0.80, 0.18)}
rows = []
for key, r in SR.items():
    diffs = {"danger - uncertainty": r.danger - r.uncertainty, "uncertainty - safety": r.uncertainty - r.safety,
             "danger - safety": r.danger - r.safety}
    row = {"population": key[0], "0/0 trials": key[1], "per rat over": key[2]}
    ok = True
    for name, d in diffs.items():
        m, lo, hi = vp.boot_ci(d.values, rng)
        row[name] = f"{m:.3f} [{lo:.3f}, {hi:.3f}]"
        ok &= lo > 0 and abs(m - expected[name][0]) <= expected[name][1]
    row["criterion met"] = ok
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Under every reading the three intervals over the 14 rats exclude zero and the means are within tolerance: in the main reading (groups A, 0/0 trials dropped, sessions counted once) danger - uncertainty 0.290, uncertainty - safety 0.512, danger - safety 0.802 (paper 0.28, 0.52, 0.80).

## Claim fig2-excited-inhibited-split

> Of the 257 cue-responsive neurons, 131 (about 51%) increased firing to danger (cue-excited) and 126
> (about 49%) decreased it (cue-inhibited).

Danger firing in the first 1 s and the last 5 s of the cue, relative to the 10 s baseline, averaged
into one value; positive is cue-excited (Results). The criterion's four readings: baseline-subtracted
Hz or normalized firing (O1 a, three-cue pool), and the two window means averaged or the 6 s
pooled. The authors' code sums the two window means of its own per-trial normalization
(`vpAnalysis.m`, lines 1182-1205), shown as a variant.

In [ ]:
def split_values(us, measure, windows, pool="3"):
    out = []
    for u in us:
        d = u["types"] == vp.DANGER
        if measure == "hz":
            on = u["onset1"][d].mean() - u["base10"][d].mean()
            late = u["last5"][d].mean() - u["base10"][d].mean()
        else:
            z = vp.normalize_cue(u, pool)["D"]
            on, late = z[vp.ONSET].mean(), z[vp.LATE].mean()
        out.append((on + late) / 2 if windows == "two means" else (on * 1 + late * 5) / 6)
    return np.array(out)

rows = []
SPLIT = {}
for pop in ("code (floored rates)", "methods"):
    us = [u for u, m in zip(units, POP[pop]) if m]
    for measure, pool in (("hz", "3"), ("z", "3"), ("z", "4"), ("z", "code")):
        for windows in ("two means", "pooled 6 s"):
            v = split_values(us, measure, windows, pool)
            SPLIT[(pop, measure, pool, windows)] = v
            rows.append({"population": pop, "firing": measure if measure == "hz" else f"normalized ({pool})",
                         "windows": windows, "excited": int((v > 0).sum()), "inhibited": int((v < 0).sum()),
                         "zero": int((v == 0).sum()),
                         "within tolerance": abs((v > 0).sum() - 131) <= 13.1 and abs((v < 0).sum() - 126) <= 12.6})
print(pd.DataFrame(rows).to_string(index=False))

How far the groups used from here on (the authors' split of the 257 neurons, A1) are from the
claims-file reading that gives counts within tolerance on the same neurons (Hz, two window means),
and from groups B (the Methods' 271 neurons split by normalized firing, three-cue pool, two means).

In [ ]:
lab_code = np.where(SPLIT[("code (floored rates)", "z", "code", "two means")] > 0, "excited", "inhibited")
lab_hz = np.where(SPLIT[("code (floored rates)", "hz", "3", "two means")] > 0, "excited", "inhibited")
print(pd.crosstab(pd.Series(lab_code, name="authors' split (A1)"), pd.Series(lab_hz, name="Hz, two means")).to_string())
print(f"neurons in the other group: {(lab_code != lab_hz).sum()} of {len(lab_code)}")
cr_code_names = [u["name"] for u, m in zip(units, POP["code (floored rates)"]) if m]
cr_meth_names = [u["name"] for u, m in zip(units, POP["methods"]) if m]
lab_b = dict(zip(cr_meth_names, np.where(SPLIT[("methods", "z", "3", "two means")] > 0, "excited", "inhibited")))
both = pd.DataFrame({"A1": [l for n, l in zip(cr_code_names, lab_code)],
                     "B": [lab_b.get(n, "not in B") for n in cr_code_names]})
print("\ngroups A (rows) against groups B (columns) on the 257 neurons:")
print(pd.crosstab(both.A1, both.B).to_string())
print(f"neurons of B not in A: {len(set(cr_meth_names) - set(cr_code_names))}")


**Result: reproduced, resting on the 257 neurons of the authors' screen (assumption A1).** On those neurons, with the three-cue pool, three of the criterion's four readings give counts within tolerance: baseline-subtracted Hz with the two window means averaged (130 excited, 127 inhibited) and normalized firing with either way of combining the windows (125/132 and 121/136). With the four-type pool two of four do: Hz with two means and normalized firing with two means (124/133). The authors' normalization gives 131/126, the paper's counts. On the 271 neurons of the Methods' unfloored screen, no reading of the criterion is within tolerance (142 to 155 inhibited, tolerance 113 to 139). The groups used from here on (the authors' split, A1) and the Hz two-means reading differ on 25 of the 257 neurons (13 cue-excited in A1 are cue-inhibited under Hz, 12 the other way). Groups A and groups B share 108 cue-excited and 115 cue-inhibited neurons; 17 of the 257 are not cue-responsive under the Methods' screen and 31 of the 271 are not among the 257.


The neuron groups used from here on (assumption A1): the authors' code's 257 neurons, split by its
normalization (131 cue-excited, 126 cue-inhibited). The variant groups (B) take the Methods'
rates (271 neurons) and the Methods' normalization with the three-cue pool, two window means.

In [ ]:
cr_code = [u for u, m in zip(units, POP["code (floored rates)"]) if m]
v_code = SPLIT[("code (floored rates)", "z", "code", "two means")]
cr_meth = [u for u, m in zip(units, POP["methods"]) if m]
v_meth = SPLIT[("methods", "z", "3", "two means")]
GROUPS = {"A": {"exc": [u for u, v in zip(cr_code, v_code) if v > 0],
                "inh": [cr_code[i] for i in np.argsort(v_code, kind="stable") if v_code[i] < 0]},
          "B": {"exc": [u for u, v in zip(cr_meth, v_meth) if v > 0],
                "inh": [cr_meth[i] for i in np.argsort(v_meth, kind="stable") if v_meth[i] < 0]}}
for g, d in GROUPS.items():
    print(f"groups {g}: cue-excited {len(d['exc'])}, cue-inhibited {len(d['inh'])}, "
          f"rats {len({u['rat'] for u in d['exc'] + d['inh']})}")

## Claim ancova-excited-no-baseline-effect

> For cue-excited neurons, baseline firing rate did not inform the cue firing pattern; an ANCOVA
> found no baseline x cue x interval interaction.

Repeated measures ANCOVA on normalized firing, cue (3) x interval (56 bins of 250 ms, 2 s before
cue onset to 2 s after offset), covariate the baseline rate (Hz, 10 s before cue onset). A claim of
absence: reproduced when p >= 0.05 for the three-way interaction. Readings O1 (a) and (b); the
authors' normalization as a variant.

In [ ]:
def cue_array(us, pool):
    n = [vp.normalize_cue(u, pool) for u in us]
    return np.stack([vp.stack(n, c) for c in ("D", "U", "S")], 1)       # (neurons, 3, 200)

def ancova_rows(group_key):
    rows = []
    for g in ("A", "B"):
        us = GROUPS[g][group_key]
        base = np.array([u["base10"].mean() for u in us])
        for pool in ("3", "4", "code"):
            Y = cue_array(us, pool)[:, :, vp.ANOVA_BINS]
            r = vp.rm_anova(Y, ["cue", "interval"], covariate=base)["cov x cue x interval"]
            rows.append({"groups": g, "pool (O1)": pool, "n": len(us), "F": r[0], "df": f"{r[1]},{r[2]}",
                         "p": f"{r[3]:.3g}", "eta2p": r[4]})
    return pd.DataFrame(rows)

print(ancova_rows("exc").round(4).to_string(index=False))

**Result: reproduced on the paper's cue-excited neurons (groups A).** The baseline x cue x interval interaction is not significant under either pool (p = 0.235 and 0.182; the authors' normalization p = 0.187; paper F(110,14080) = 1.12, p = 0.18, whose degrees of freedom imply 130 neurons, one fewer than the 131 used here). On groups B (the Methods' screen and split, 123 cue-excited neurons) the interaction is significant (p = 0.0006 and 0.0001).

## Claim ancova-inhibited-baseline-effect

> For cue-inhibited neurons, baseline firing rate did inform the cue firing pattern; the ANCOVA
> found a significant baseline x cue x interval interaction.

In [ ]:
print(ancova_rows("inh").round(6).to_string(index=False))

**Result: reproduced.** The baseline x cue x interval interaction is significant for cue-inhibited neurons under every pool and both groupings (groups A, three-cue pool, F(110,13640) = 1.56, p = 0.00015; paper F(110,13420) = 1.93, p = 2.34e-8).

## Claim kmeans-three-clusters

> K-means clustering of cue-inhibited neurons on baseline firing rate, coefficient of variance,
> coefficient of skewness, waveform half duration and amplitude ratio gave three clusters differing
> mainly in baseline firing rate, Low (74), Intermediate (34) and High (18) firing; four of the five
> features differed across clusters, baseline firing rate most, and amplitude ratio did not.

The criterion's readings: CV and skewness over (a) the whole session or (b) the 10 s baselines;
k-means on (a) raw or (b) Z-scored features (O7), 100 restarts with a fixed seed (the claims file's
standard choice). The Methods name Matlab's `kmeans`, whose default is one k-means++ start. The
authors' code (`vpAnalysis.m`, lines 1455-1487) runs it once after `rng(1)` on raw features, with
CV and skewness over the 20 s before each cue and the floored baseline rate; `vp.matlab_kmeans`
repeats that run. The table gives each partition's within-cluster sum of squares (SSE) on its own
features.

In [ ]:
def cluster_anova(F, lab):
    names = ["baseline rate", "CV", "skewness", "half duration", "amplitude ratio"]
    return {n: stats.f_oneway(*[F[lab == k, j] for k in range(3)]) for j, n in enumerate(names)}

def feature_matrix(us, spikes_from):
    return np.array([[u["base10"].mean(), *vp.isi_stats(u, spikes_from), *vp.waveform_features(u["wave"])] for u in us])

def km_row(label, F, lab, us):
    a = cluster_anova(F, lab)
    cnt = np.bincount(lab, minlength=3)
    fs = {k: v.statistic for k, v in a.items()}
    ok_counts = abs(cnt[0] - 74) <= 7.4 and abs(cnt[1] - 34) <= 3.4 and abs(cnt[2] - 18) <= 1.8
    ok_feat = (all(a[k].pvalue < 0.05 for k in ["baseline rate", "CV", "skewness", "half duration"])
               and a["amplitude ratio"].pvalue >= 0.05 and max(fs, key=fs.get) == "baseline rate")
    return {"reading": label, "Low": cnt[0], "Int": cnt[1], "High": cnt[2], "SSE": vp.kmeans_sse(F, lab),
            **{f"F {k}": f"{v.statistic:.2f} (p {v.pvalue:.2g})" for k, v in a.items()},
            "counts ok": ok_counts, "features ok": ok_feat}

CLUSTERS = {}
rows = []
for g in ("A", "B"):
    inh = GROUPS[g]["inh"]
    for spk in ("session", "base10"):
        F = feature_matrix(inh, spk)
        for scale in (False, True):
            lab = vp.kmeans3(F, scale)
            label = f"{g}: CV over {spk}, {'Z-scored' if scale else 'raw'}, 100 restarts"
            CLUSTERS[label] = lab
            rows.append(km_row(label, F, lab, inh))
        lab1, _ = vp.matlab_kmeans(F, seed=1)
        label = f"{g}: CV over {spk}, raw, one start after rng(1)"
        CLUSTERS[label] = lab1
        rows.append(km_row(label, F, lab1, inh))
    Fc = np.array([vp.code_features(u) for u in inh])
    lab, sse = vp.matlab_kmeans(Fc, seed=1)
    label = f"{g}: authors' code, one start after rng(1)"
    CLUSTERS[label] = lab
    rows.append(km_row(label, Fc, lab, inh))
    lab100 = vp.kmeans3(Fc, False)
    CLUSTERS[f"{g}: authors' code features, 100 restarts"] = lab100
    rows.append(km_row(f"{g}: authors' code features, 100 restarts", Fc, lab100, inh))
km = pd.DataFrame(rows)
print(km.round(1).to_string(index=False))

How often one Matlab-style start (the Methods' `kmeans` with its defaults) reaches each partition,
over seeds 1 to 300, on the authors' features of groups A:

In [ ]:
Fc = np.array([vp.code_features(u) for u in GROUPS["A"]["inh"]])
seen = Counter()
for s in range(1, 301):
    lab, sse = vp.matlab_kmeans(Fc, seed=s)
    seen[(tuple(int(x) for x in np.bincount(lab, minlength=3)), round(sse, 1))] += 1
print(pd.DataFrame([{"Low, Int, High": k[0], "SSE": k[1], "seeds": v} for k, v in seen.most_common()]).to_string(index=False))

Do the clusters of the claims file's reading (ISIs over the 10 s baselines, raw features, one
start after `rng(1)`) have the same members as those of the authors' features? And how far is the
lower-SSE partition of 100 restarts from them? Adjusted Rand index, 1 for identical partitions.

In [ ]:
from sklearn.metrics import adjusted_rand_score
lab_code_km = CLUSTERS["A: authors' code, one start after rng(1)"]
for other in ("A: CV over base10, raw, one start after rng(1)", "A: authors' code features, 100 restarts",
              "A: CV over base10, raw, 100 restarts"):
    print(f"authors' code clusters (rows) against {other} (columns), adjusted Rand index "
          f"{adjusted_rand_score(lab_code_km, CLUSTERS[other]):.3f}")
    print(pd.crosstab(pd.Series(lab_code_km, name="code"), pd.Series(CLUSTERS[other], name="other")).to_string(), "\n")


**Result: reproduced, resting on the k-means start (assumption A2).** With one k-means++ start after rng(1), the Methods' Matlab `kmeans` with its defaults as the authors' code runs it, the 126 cue-inhibited neurons of groups A fall into Low 74, Intermediate 34 and High 18, both with the claims file's reading (b) for the ISIs (10 s baselines) and with the authors' features, and the two partitions have the same members (adjusted Rand index 1.000); with the authors' features the five ANOVAs give the paper's F values to the printed digits (546.73, 8.79, 18.20, 17.90, 2.12), and amplitude ratio is the one feature with p >= 0.05. This partition is a local optimum: with 100 restarts (the claims file's standard choice) k-means finds partitions with a lower within-cluster sum of squares, 58/46/22 or 56/48/22 on groups A and 74/54/20 or 76/52/20 on groups B, none within tolerance. The lower-SSE partition of groups A moves 16 Low neurons to Intermediate and 4 Intermediate neurons to High (adjusted Rand index 0.579). Over 300 single starts the paper's partition is reached in 98. So the seed changes the clusters, against the claims file's statement that it cannot change a conclusion beyond the edge of an interval, and every cluster claim rests on A2. Whether the claims file's reading should be Matlab's default of one start is asked of a reader (questions/crcns-bf-3.yaml). On groups B no reading gives counts within tolerance.

The clusters used from here on are those of the authors' code on groups A (Low 74, Intermediate 34,
High 18; assumption A2). The variant clusters (B) are those of groups B under the claims file's
reading (a) for both details (CV over the session, raw features, 100 restarts).

In [ ]:
lab_A = CLUSTERS["A: authors' code, one start after rng(1)"]
lab_B = CLUSTERS["B: CV over session, raw, 100 restarts"]
for g, lab in (("A", lab_A), ("B", lab_B)):
    for k, name in enumerate(("low", "int", "high")):
        GROUPS[g][name] = [u for u, l in zip(GROUPS[g]["inh"], lab) if l == k]
    print(g, {k: len(v) for k, v in GROUPS[g].items()})

## Claim clusters-differ-in-cue-firing

> The three cue-inhibited clusters differed in patterned cue firing; each pair gave a significant
> cluster x cue x interval interaction.

In [ ]:
rows = []
for g in ("A", "B"):
    for pool in ("3", "4", "code"):
        for a_name, b_name in (("low", "int"), ("low", "high"), ("int", "high")):
            us = GROUPS[g][a_name] + GROUPS[g][b_name]
            Y = cue_array(us, pool)[:, :, vp.ANOVA_BINS]
            grp = np.array([0] * len(GROUPS[g][a_name]) + [1] * len(GROUPS[g][b_name]))
            r = vp.rm_anova(Y, ["cue", "interval"], between=grp, between_name="cluster")["cluster x cue x interval"]
            rows.append({"groups": g, "pool (O1)": pool, "pair": f"{a_name} vs {b_name}", "F": r[0],
                         "df": f"{r[1]},{r[2]}", "p": f"{r[3]:.3g}", "p < 0.005": r[3] < 0.005})
cd = pd.DataFrame(rows)
print(cd.round(3).to_string(index=False))

**Result: partially reproduced.** On the paper's clusters (groups A) with the Methods' normalization, the cluster x cue x interval interaction is significant for every pair at p < 0.05, but Low vs Intermediate does not reach the paper's p < 0.005 (p = 0.035 three-cue pool, 0.011 four-type pool); the other two pairs do (p <= 0.0006). With the authors' normalization all three pairs give p < 0.005 (Low vs Intermediate p = 0.0027), a variant outside the claims file. The clusters of groups B also give p < 0.005 for all pairs, but they are not the paper's clusters (Intermediate 54 neurons) and do not count towards the outcome.

## Claim clusters-by-rat

> Low firing neurons came from 11 of 14 rats and Intermediate firing neurons from 9 of 14, while
> High firing neurons came from only 5 rats, 11 of the 18 from one rat (PA02).

In [ ]:
rows = []
for label, lab in CLUSTERS.items():
    g = label[0]
    inh = GROUPS[g]["inh"]
    rats = [Counter(u["rat"] for u, l in zip(inh, lab) if l == k) for k in range(3)]
    ok = (abs(len(rats[0]) - 11) <= 1 and abs(len(rats[1]) - 9) <= 1 and abs(len(rats[2]) - 5) <= 1
          and abs(rats[2].get("pa02", 0) - 11) <= 1.1)
    rows.append({"clusters": label, "rats Low": len(rats[0]), "rats Int": len(rats[1]), "rats High": len(rats[2]),
                 "High from pa02": rats[2].get("pa02", 0), "High neurons": sum(rats[2].values()), "within tolerance": ok})
print(pd.DataFrame(rows).to_string(index=False))
print("\nneurons per rat in each cluster, main clusters:")
print(pd.DataFrame({n: Counter(u["rat"] for u in GROUPS["A"][n]) for n in ("low", "int", "high")}).fillna(0).astype(int).T.to_string())

**Result: reproduced, resting on the k-means start (assumption A2).** On the main clusters, Low firing
neurons come from 11 rats, Intermediate from 9 and High from 5, with 11 of the 18 High firing neurons
from rat pa02, the paper's counts. With 100 restarts the High cluster of groups A has 22 neurons, 14 of
them from pa02, outside tolerance.

## Shared measures for the cue-firing claims

Normalized firing per cue and neuron (Methods, Z-score normalization) under readings O1 (a)
three-cue pool and (b) four trial types, with the authors' per-trial normalization as a variant.
Repeated measures ANOVA, cue (3) x interval (56 bins). Period means per neuron: onset 0-1 s, late
cue 5-10 s, delay 10-12 s, with studentized bootstrap intervals over neurons.

In [ ]:
PERIODS = {"onset": vp.ONSET, "late cue": vp.LATE, "delay": vp.DELAY}

def cue_anova_table(group_key, groups=("A", "B"), pools=("3", "4", "code")):
    rows = []
    for g in groups:
        us = GROUPS[g][group_key]
        for pool in pools:
            Z = cue_array(us, pool)
            r = vp.rm_anova(Z[:, :, vp.ANOVA_BINS], ["cue", "interval"])
            cue_mean = Z[:, :, 80:120].mean((0, 2))
            onset_mean = Z[:, :, vp.ONSET].mean((0, 2))
            rows.append({"groups": g, "pool": pool, "n": len(us),
                         **{k: f"F({r[k][1]},{r[k][2]}) = {r[k][0]:.2f}, p = {r[k][3]:.2g}" for k in ("cue", "interval", "cue x interval")},
                         "cue eta2p": round(r["cue"][4], 3), "all p < 0.05": all(r[k][3] < 0.05 for k in ("cue", "interval", "cue x interval")),
                         "10 s cue mean D/U/S": "/".join(f"{x:.3f}" for x in cue_mean),
                         "onset mean D/U/S": "/".join(f"{x:.3f}" for x in onset_mean)})
    return pd.DataFrame(rows)

def period_values(us, pool):
    Z = cue_array(us, pool)
    return {(c, p): Z[:, i, idx].mean(1) for i, c in enumerate("DUS") for p, idx in PERIODS.items()}

def period_cis(group_key, rng, groups=("A", "B"), pools=("3", "4", "code")):
    out = {}
    for g in groups:
        for pool in pools:
            pv = period_values(GROUPS[g][group_key], pool)
            res = {}
            for p in PERIODS:
                for c in "DUS":
                    res[(c, p)] = vp.boot_ci(pv[(c, p)], rng)
                res[("D-U", p)] = vp.boot_ci(pv[("D", p)] - pv[("U", p)], rng)
                res[("U-S", p)] = vp.boot_ci(pv[("U", p)] - pv[("S", p)], rng)
            out[(g, pool)] = res
    return out

def show_cis(cis):
    rows = []
    for (g, pool), res in cis.items():
        for (c, p), (m, lo, hi) in res.items():
            rows.append({"groups": g, "pool": pool, "measure": c, "period": p, "mean": m, "ci_low": lo, "ci_high": hi})
    df = pd.DataFrame(rows)
    return df.pivot_table(index=["measure", "period"], columns=["groups", "pool"],
                          values=["mean", "ci_low", "ci_high"]).round(3)

def judge(res, rules):
    """rules: list of (key, kind, expected, tol); kind 'below', 'above' (the interval excludes zero
    on that side) or 'contains'; expected may be None. Returns (pattern holds, values within
    tolerance)."""
    pattern = values = True
    for key, kind, exp, tol in rules:
        m, lo, hi = res[key]
        pattern &= {"below": hi < 0, "above": lo > 0, "contains": lo <= 0 <= hi}[kind]
        if exp is not None:
            values &= abs(m - exp) <= tol
    return bool(pattern), bool(values)

## Which part of the normalization separates the Methods from the authors' code

The Methods' normalization gives period means about twice the paper's; the authors' code gives the
paper's values. The two differ in three elements: Z over the trial-averaged curves (Methods) or over
single trials (code), a 10 s (Methods) or 2 s (code) baseline, and flooring the baseline rate
(code). Each combination, three-cue pool, groups A, against the paper's means (Fig. 3b, 3d, 6b).

In [ ]:
NV = {}
rows = []
for z_over in ("trial means", "single trials"):
    for base_s in (10, 2):
        for floor in (False, True):
            r = {"Z over": z_over, "baseline": f"{base_s} s", "floored": floor}
            for gk in ("low", "int", "exc"):
                n = [vp.normalize_variant(u, z_over, base_s, floor) for u in GROUPS["A"][gk]]
                D, U = vp.stack(n, "D"), vp.stack(n, "U")
                if gk == "low":
                    r |= {"Low D onset": D[:, vp.ONSET].mean(), "Low U onset": U[:, vp.ONSET].mean(),
                          "Low D late": D[:, vp.LATE].mean(), "Low U late": U[:, vp.LATE].mean()}
                elif gk == "int":
                    r |= {"Int D onset": D[:, vp.ONSET].mean(), "Int U onset": U[:, vp.ONSET].mean()}
                else:
                    r |= {"Exc D-U onset": (D - U)[:, vp.ONSET].mean()}
            rows.append(r)
nv = pd.DataFrame(rows)
paper = {"Low D onset": (-0.37, 0.08), "Low U onset": (-0.14, 0.08), "Low D late": (-0.27, 0.08), "Low U late": (-0.09, 0.06),
         "Int D onset": (-0.54, 0.15), "Int U onset": (-0.25, 0.17), "Exc D-U onset": (0.26, 0.08)}
nv["within tolerance"] = [sum(abs(r[k] - v) <= t for k, (v, t) in paper.items()) for _, r in nv.iterrows()]
print(nv.round(3).to_string(index=False))
print("paper (tolerance):", {k: f"{v} ({t})" for k, (v, t) in paper.items()})
print("check: trial means, 10 s, not floored equals the Methods' normalization (three-cue pool):",
      np.allclose(vp.stack([vp.normalize_variant(u, "trial means", 10, False) for u in GROUPS["A"]["low"]], "D"),
                  vp.stack([vp.normalize_cue(u, "3") for u in GROUPS["A"]["low"]], "D")))


**Z over single trials in place of trial means accounts for the difference.** With Z over single
trials all seven means are within the paper's tolerance under every baseline and floor (for
example Low danger onset -0.344 with the 10 s baseline unfloored, paper -0.37); with Z over trial
means none is (-0.709). Flooring moves each mean by at most 0.008. The baseline length moves Intermediate uncertainty
onset by 0.15 with Z over trial means (-0.592 to -0.442) and by 0.077 with Z over single trials
(-0.330 to -0.253); it moves the other six means by at most 0.062 and the four Low means by at
most 0.057. The
Methods describe Z over trial means ("Mean differential firing was Z-score normalized across all
trial types within a single unit"); the paper's printed values follow the code.

## Claim fig3a-low-differential-cue-firing

> Low firing neurons showed the greatest inhibition to danger, modest inhibition to uncertainty and
> none to safety, throughout the cue; the ANOVA gave main effects of cue and interval and a cue x
> interval interaction.

Reproduced when cue, interval and cue x interval all have p < 0.05 and the 10 s cue means order
danger < uncertainty < safety.

In [ ]:
t = cue_anova_table("low")
t["criterion met"] = t["all p < 0.05"] & t["10 s cue mean D/U/S"].map(lambda s: (lambda a: a[0] < a[1] < a[2])([float(x) for x in s.split("/")]))
print(t.to_string(index=False))

**Result: reproduced.** For the 74 Low firing neurons, cue, interval and cue x interval are all significant under every reading (three-cue pool F(2,146) = 32.31 for cue, paper F(2,142) = 23.45), and the 10 s cue means order danger (-0.561) < uncertainty (-0.224) < safety (0.050).

## Claim fig3b-low-inhibited-to-threat-cues

> Low firing neurons were inhibited below baseline to danger and uncertainty at cue onset and in the
> late cue period, but not to safety.

In [ ]:
CIS_LOW = period_cis("low", seed_for("fig3b-low"))
print(show_cis(CIS_LOW).to_string())
rules = [(("D", "onset"), "below", -0.37, 0.08), (("U", "onset"), "below", -0.14, 0.08),
         (("D", "late cue"), "below", -0.27, 0.08), (("U", "late cue"), "below", -0.09, 0.06),
         (("S", "onset"), "contains", None, 0), (("S", "late cue"), "contains", None, 0)]
print("\n(pattern holds, values within tolerance):", {k: judge(v, rules) for k, v in CIS_LOW.items()})

**Result: partially reproduced.** With the four-type pool the pattern holds: danger and uncertainty intervals exclude zero below at onset and late cue and both safety intervals contain zero (with the three-cue pool the late-cue safety interval lies just above zero, [0.001, 0.216]). The means are about twice the paper's, outside tolerance (four-type pool: danger onset -0.614, uncertainty onset -0.299, danger late -0.427, uncertainty late -0.209; paper -0.37, -0.14, -0.27, -0.09). With the authors' per-trial normalization (variant) the pattern holds and all four means are within tolerance (-0.364, -0.138, -0.266, -0.094). The section before this claim shows that Z over single trials, the one element of the code's normalization that matters, is what brings the means to the paper's.


## Claim fig3b-low-differential

> Low firing neurons fired differentially to danger and uncertainty at onset and late cue;
> uncertainty and safety firing were similar at onset but differed during late cue and delay.

The intervals are those printed in the previous section (same bootstrap).

In [ ]:
rules = [(("D-U", "onset"), "below", -0.23, 0.10), (("D-U", "late cue"), "below", -0.17, 0.08),
         (("U-S", "onset"), "contains", -0.08, 0.11), (("U-S", "late cue"), "below", -0.14, 0.11),
         (("U-S", "delay"), "below", -0.12, 0.14)]
for k, v in CIS_LOW.items():
    print(k, {f"{a} {b}": ci_text(*v[(a, b)]) for (a, b), *_ in rules}, "(pattern, values):", judge(v, rules))

**Result: partially reproduced.** With the four-type pool the pattern holds (danger - uncertainty below zero at onset and late cue, uncertainty - safety containing zero at onset and below zero in late cue and delay), but the danger - uncertainty means are outside tolerance (-0.315 and -0.218; paper -0.23 and -0.17); with the three-cue pool the onset uncertainty - safety interval excludes zero ([-0.378, -0.003]). With the authors' normalization the means match the paper (-0.226, -0.172, -0.082, -0.144, -0.122) but the delay interval reaches zero ([-0.232, 0.002]; the paper's upper bound was -6.3e-4).

## Claim fig3c-intermediate-differential-cue-firing

> Intermediate firing neurons showed the greatest inhibition to danger at cue onset, less to
> uncertainty and least to safety, with inhibition specific to danger and uncertainty for the rest
> of the cue; the ANOVA gave main effects of cue and interval and a cue x interval interaction.

In [ ]:
t = cue_anova_table("int")
t["criterion met"] = t["all p < 0.05"] & t["onset mean D/U/S"].map(lambda s: (lambda a: a[0] < a[1] < a[2])([float(x) for x in s.split("/")]))
print(t.to_string(index=False))

**Result: reproduced.** For the 34 Intermediate firing neurons, cue, interval and cue x interval are significant under every reading (three-cue pool F(2,66) = 23.81; paper F(2,66) = 27.25), and onset means order danger (-0.974) < uncertainty (-0.592) < safety (-0.337).

## Claim fig3d-intermediate-inhibition

> Intermediate firing neurons were inhibited to all three cues at onset but only to danger and
> uncertainty during late cue.

In [ ]:
CIS_INT = period_cis("int", seed_for("fig3d-int"))
print(show_cis(CIS_INT).to_string())
rules = [(("D", "onset"), "below", -0.54, 0.15), (("U", "onset"), "below", -0.25, 0.17),
         (("S", "onset"), "below", -0.21, 0.10), (("D", "late cue"), "below", -0.49, 0.13),
         (("U", "late cue"), "below", -0.23, 0.11), (("S", "late cue"), "contains", None, 0)]
print("\n(pattern holds, values within tolerance):", {k: judge(v, rules) for k, v in CIS_INT.items()})

**Result: partially reproduced.** Under both pools the pattern holds (all three cues below zero at onset, danger and uncertainty below zero in late cue, the late-cue safety interval containing zero), but the means are outside tolerance (three-cue pool: danger onset -0.974, paper -0.54). With the authors' normalization both hold (-0.544, -0.253, -0.207, -0.488, -0.234; paper -0.54, -0.25, -0.21, -0.49, -0.23).

## Claim fig3d-intermediate-differential

> Intermediate firing neurons fired differentially to danger and uncertainty at onset and late
> cue; uncertainty and safety did not differ at onset but differed during late cue.

In [ ]:
rules = [(("D-U", "onset"), "below", -0.29, 0.15), (("D-U", "late cue"), "below", -0.26, 0.16),
         (("U-S", "onset"), "contains", -0.05, 0.19), (("U-S", "late cue"), "below", -0.21, 0.16)]
for k, v in CIS_INT.items():
    print(k, {f"{a} {b}": ci_text(*v[(a, b)]) for (a, b), *_ in rules}, "(pattern, values):", judge(v, rules))

**Result: partially reproduced.** Danger - uncertainty excludes zero at onset and uncertainty - safety contains zero at onset and excludes it in late cue, but the late-cue danger - uncertainty interval contains zero under both pools ([-0.658, 0.042] and [-0.582, 0.065]). With the authors' normalization the four parts hold and the means match the paper (-0.291, -0.254, -0.046, -0.211; paper -0.29, -0.26, -0.05, -0.21).

## Claim supp5-low-cue-correlations

> In Low and Intermediate firing neurons, firing to danger and uncertainty was commonly positively
> correlated across neurons, while firing to uncertainty and safety showed zero or negative
> correlations.

Pearson correlation across neurons of period normalized firing. "Commonly" is read as at least two
of the three periods (an assumption of the claims file).

In [ ]:
def corr_table(group_keys, groups=("A", "B"), pools=("3", "4", "code")):
    rows = []
    for g in groups:
        for gk in group_keys:
            for pool in pools:
                pv = period_values(GROUPS[g][gk], pool)
                for p in PERIODS:
                    du = stats.pearsonr(pv[("D", p)], pv[("U", p)])
                    us_ = stats.pearsonr(pv[("U", p)], pv[("S", p)])
                    rows.append({"groups": g, "cluster": gk, "pool": pool, "period": p, "n": len(pv[("D", p)]),
                                 "r D-U": du.statistic, "p D-U": du.pvalue, "r U-S": us_.statistic, "p U-S": us_.pvalue})
    return pd.DataFrame(rows)

cs5 = corr_table(["low", "int"])
PERIODS_CODE = {"first 5 s": np.arange(80, 100), "late cue": vp.LATE, "delay": vp.DELAY}
print(cs5.round(4).to_string(index=False))
verdict = {}
for (g, gk, pool), d in cs5.groupby(["groups", "cluster", "pool"]):
    verdict[(g, gk, pool)] = ((d["r D-U"] > 0) & (d["p D-U"] < 0.05)).sum() >= 2 and not ((d["r U-S"] > 0) & (d["p U-S"] < 0.05)).any()
print("\nper cluster, criterion part met:", verdict)
for g in ("A", "B"):
    for pool in ("3", "4", "code"):
        print(g, pool, "criterion met (both clusters):", verdict[(g, "low", pool)] and verdict[(g, "int", pool)])
print("\nvariant: the authors' code correlates the first 5 s of cue in place of the first 1 s (vpAnalysis.m, lines 1719-1775); groups A")
rows = []
for gk in ("low", "int"):
    for pool in ("3", "code"):
        Z = cue_array(GROUPS["A"][gk], pool)
        for p, idx in PERIODS_CODE.items():
            v = {c: Z[:, i, idx].mean(1) for i, c in enumerate("DUS")}
            du, us_ = stats.pearsonr(v["D"], v["U"]), stats.pearsonr(v["U"], v["S"])
            rows.append({"cluster": gk, "pool": pool, "period": p, "r D-U": du.statistic, "p D-U": du.pvalue,
                         "r U-S": us_.statistic, "p U-S": us_.pvalue})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: not reproduced (resting on the claims file's reading of 'commonly' and on assumption A2).** Danger-uncertainty correlations are positive and significant in all three periods for Low firing neurons and in two of three for Intermediate neurons, but the uncertainty-safety correlation of Low firing neurons at cue onset is positive and significant (r = 0.36, p = 0.001, three-cue pool), under every reading. With the first 5 s of cue in place of the first 1 s, as the authors' code correlates these clusters, the onset uncertainty-safety correlation is near zero (r = 0.03, three-cue pool), so the outcome depends on which window Supplementary Fig. 5 used. Which window Supplementary Fig. 5 used is asked of a reader (questions/crcns-bf-3.yaml).


## Claim fig6a-excited-differential-cue-firing

> Cue-excited neurons increased firing sharply at cue onset, most to danger, less to uncertainty
> and least to safety, with differential firing through the cue and delay; the ANOVA gave main
> effects of cue and interval and a cue x interval interaction.

In [ ]:
t = cue_anova_table("exc")
t["criterion met"] = t["all p < 0.05"] & t["onset mean D/U/S"].map(lambda s: (lambda a: a[0] > a[1] > a[2])([float(x) for x in s.split("/")]))
print(t.to_string(index=False))

**Result: reproduced.** For the 131 cue-excited neurons, cue, interval and cue x interval are significant under every reading (three-cue pool F(2,260) = 67.31; paper F(2,258) = 68.22), and onset means order danger (1.157) > uncertainty (0.679) > safety (0.308).

## Claim fig6b-excited-period-firing

> Cue-excited neurons increased firing to all cues at onset but only to danger and uncertainty in
> late cue and delay, and fired differentially to danger vs. uncertainty and uncertainty vs. safety
> in every period.

In [ ]:
CIS_EXC = period_cis("exc", seed_for("fig6b-excited-period-firing"))
print(show_cis(CIS_EXC).to_string())
rules = [(("D", "onset"), "above", None, 0), (("U", "onset"), "above", None, 0), (("S", "onset"), "above", None, 0),
         (("D", "late cue"), "above", None, 0), (("U", "late cue"), "above", None, 0),
         (("D", "delay"), "above", None, 0), (("U", "delay"), "above", None, 0),
         (("S", "late cue"), "contains", None, 0), (("S", "delay"), "contains", None, 0),
         (("D-U", "onset"), "above", 0.26, 0.08), (("D-U", "late cue"), "above", 0.24, 0.06),
         (("D-U", "delay"), "above", 0.19, 0.07), (("U-S", "onset"), "above", 0.21, 0.09),
         (("U-S", "late cue"), "above", 0.15, 0.07), (("U-S", "delay"), "above", 0.22, 0.09)]
print("\n(pattern holds, values within tolerance):", {k: judge(v, rules) for k, v in CIS_EXC.items()})

**Result: partially reproduced.** All three cues raise firing at onset, danger and uncertainty in late cue and delay, and all six difference intervals exclude zero above it under every reading. Two parts fail: safety firing in the delay lies below zero under every reading (three-cue pool, interval [-0.254, -0.077]), and in the late cue under the three-cue pool ([-0.150, -0.004]) and the authors' normalization ([-0.098, -0.008]), where the criterion asks for intervals containing zero (the four-type pool's late-cue interval, [-0.126, 0.003], contains it), and the difference means are outside tolerance with the Methods' normalization (danger - uncertainty at onset 0.478, paper 0.26). With the authors' normalization the six means are within tolerance (0.265, 0.239, 0.195, 0.206, 0.146, 0.222) but the safety intervals still exclude zero. The paper says only that increases after onset were limited to the threat cues, so whether a safety decrease contradicts it is asked of a reader (questions/crcns-bf-3.yaml).

## Claim supp8-excited-cue-correlations

> In cue-excited neurons danger and uncertainty firing were positively correlated in all periods,
> while uncertainty and safety firing were positively correlated at onset, uncorrelated in late
> cue and negatively correlated in the delay.

In [ ]:
cs8 = corr_table(["exc"])
print(cs8.round(4).to_string(index=False))
for (g, pool), d in cs8.groupby(["groups", "pool"]):
    d = d.set_index("period")
    ok = (((d["r D-U"] > 0) & (d["p D-U"] < 0.05)).all() and d.loc["onset", "r U-S"] > 0 and d.loc["onset", "p U-S"] < 0.05
          and d.loc["late cue", "p U-S"] >= 0.05 and d.loc["delay", "r U-S"] < 0 and d.loc["delay", "p U-S"] < 0.05)
    print(g, pool, "criterion met:", ok)

**Result: partially reproduced.** On groups A, danger-uncertainty correlations are positive in every period and uncertainty-safety correlations are positive at onset (r = 0.37, p < 0.0001) and not significant in late cue, but the delay correlation, though negative, is not significant under the Methods' normalization (r = -0.08, p = 0.34). It is significant with the authors' normalization (r = -0.23, p = 0.008) and on groups B (r = -0.23, p = 0.012).

## Nose-poke pauses (claims supp6-low-nosepoke-pauses and supp9-excited-nosepoke-pauses)

Firing around the last poke before a pause in poking, outside cues, in 16 bins of 250 ms from 2 s
before to 2 s after; pre = the 500 ms before, post = the first 500 ms after (Methods). Pause
readings from the claims file: no poke for (a) 10 s or (b) 5 s, outside the trial window and within
recorded time. Normalization as reward (O4): each bin minus the mean of the 1 s before, then (i) Z
over the 16 bins of the neuron's mean, or (ii) divided by the SD of the neuron's cue normalization
(three-cue pool). The authors' code (variant): gaps of more than 2 s between pokes 18 to 2 s before
a cue onset, baseline the first 1 s of the window, Z over single events (`cubeScriptPR25.m`).

In [ ]:
def event_matrix(us, kind, reading, scale):
    rows, kept = [], []
    for u in us:
        ev = vp.reward_events(u, reading) if kind == "reward" else vp.pause_events(u, reading)
        sd = vp.normalize_cue(u, "3")["sd"]
        f = vp.event_firing(u, ev, scale, sd)
        if f is not None:
            rows.append(f)
            kept.append(u)
    return np.array(rows), kept

EVENT_READINGS = [("10s", "i"), ("10s", "ii"), ("5s", "i"), ("5s", "ii"), ("code", "i"), ("code", "ii"), ("code", "code")]

def pause_rows(group_keys, rng):
    rows = []
    for gk in group_keys:
        for reading, scale in EVENT_READINGS:
            M, kept = event_matrix(GROUPS["A"][gk], "pause", reading, scale)
            n_ev = np.array([len(vp.pause_events(u, reading)) for u in GROUPS["A"][gk]])
            r = vp.rm_anova(M, ["interval"])["interval"]
            pre, post = M[:, 6:8].mean(1), M[:, 8:10].mean(1)
            ci_post, ci_diff = vp.boot_ci(post, rng), vp.boot_ci(post - pre, rng)
            rows.append({"cluster": gk, "pause": reading, "scale": scale, "neurons with events": f"{len(kept)} of {len(GROUPS['A'][gk])}",
                         "events": int(n_ev.sum()),
                         "events per neuron (min, quartiles, max)": "{:.0f} / {:.0f} / {:.0f} / {:.0f} / {:.0f}".format(*np.percentile(n_ev, [0, 25, 50, 75, 100])),
                         "interval F": f"F({r[1]},{r[2]}) = {r[0]:.2f}, p = {r[3]:.2g}",
                         "post": ci_text(*ci_post), "post - pre": ci_text(*ci_diff), "_post": ci_post, "_diff": ci_diff})
    return pd.DataFrame(rows)

### Claim supp6-low-nosepoke-pauses

> Pauses in nose poking during the inter-trial interval, in the absence of cues, did not inhibit Low
> and Intermediate firing neurons, so their cue inhibition is not simply the cessation of nose poking.

In [ ]:
p6 = pause_rows(["low", "int"], seed_for("supp6-low-nosepoke-pauses"))
p6["no inhibition (post and post - pre not below zero)"] = [not (a[2] < 0) and not (b[2] < 0) for a, b in zip(p6._post, p6._diff)]
print(p6.drop(columns=["_post", "_diff"]).to_string(index=False))

**Result: partially reproduced, resting on the definition of a pause (assumption A3).** Low firing neurons show no inhibition after a pause under any reading. For Intermediate neurons the absence holds with the authors' pause definition (more than 2 s without a poke, 18 to 2 s before a cue) and the claims file's scalings (post -0.252 [-0.572, 0.062] and -0.024 [-0.212, 0.249]), but with the authors' own single-event scaling Intermediate post-pause firing lies below zero (-0.103 [-0.201, -0.020]), and so it does with 5 s pauses and scaling (ii) (-0.399 [-0.646, -0.051], post - pre [-0.613, -0.016]). The claims file's 10 s pauses also show no inhibition, but there only 14 of 34 Intermediate neurons have a pause (25 events in all, median 0 per neuron), too few to support an absence. The absence holds in the Low cluster and only under some readings in the Intermediate cluster. With the authors' pauses the interval effect is significant for Intermediate neurons under every scaling (p = 1.2e-5 to 3.3e-8) and not for Low neurons (p >= 0.13).


### Claim supp9-excited-nosepoke-pauses

> Cessation of nose poking in the absence of cues did not increase firing of cue-excited neurons.

In [ ]:
p9 = pause_rows(["exc"], seed_for("supp9-excited-nosepoke-pauses"))
p9["no increase (post and post - pre not above zero)"] = [not (a[1] > 0) and not (b[1] > 0) for a, b in zip(p9._post, p9._diff)]
print(p9.drop(columns=["_post", "_diff"]).to_string(index=False))

**Result: reproduced, resting on the definition of a pause (assumption A3).** With the authors' pause definition (3367 events, 9 to 46 per neuron), neither interval lies above zero for cue-excited neurons under either scaling of the claims file or the authors' scaling (post - pre [-0.039, 0.361], [-0.013, 0.222], [-0.001, 0.112]), although the interval effect is significant (p = 8.3e-8 to 1.6e-6). The 10 s pauses agree but rest on 71 of 131 neurons (135 events, median 1 per neuron). With 5 s pauses and scaling (ii), firing rises after the pause (post 0.172 [0.037, 0.336], post - pre [0.052, 0.389]), so the outcome depends on the pause definition and rests on A3.


## Single-unit regression (claims fig4*, fig6-excited-assignment-050, fig6c, fig6d)

Per neuron, the 16 trials, 14 bins of 1 s from 2 s before cue onset to 2 s after offset;
normalized firing regressed (least squares, a constant plus the regressors, as Matlab `regress`) on
relative threat (danger 1, safety 0, uncertainty the assignment) and the trial's suppression ratio
(fear output). Readings: per-trial firing O2 (a), (b), (c); 0/0 suppression ratios O3 dropped or 0.
Variant: the authors' code (per-trial firing as `vpAnalysis.m` lines 1112-1170, 0/0 ratios set to
1). Two-regressor model of the Low and Intermediate claims: (a) constant + relative threat at 0.25
and at 0.50 (as `vpAnalysis.m` line 1158), (b) the same plus fear output, (c) the 0.25 and 0.50
betas of the single-assignment models. Cue bins are 0-10 s (indices 2-11); early 0-4 s (2-5);
late 8-12 s (10-13).

In [ ]:
READINGS = [(o2, o3) for o2 in ("a", "b", "c") for o3 in ("drop", "zero")] + [("code", "one")]
CUE_BINS, EARLY, LATE_B = np.arange(2, 12), np.arange(2, 6), np.arange(10, 14)

def regression(us, o2, o3):
    single, two = [], {"a": [], "b": [], "c": []}
    for u in us:
        norm = vp.normalize_cue(u, "3")
        y = vp.trial_firing(u, o2, norm)
        sr = vp.fear_output(u, o3)
        single.append([vp.regress(y, [vp.threat(u["types"], a), sr]) for a in vp.ASSIGNMENTS])
        t25, t50 = vp.threat(u["types"], 0.25), vp.threat(u["types"], 0.5)
        two["a"].append(vp.regress(y, [t25, t50]))
        two["b"].append(vp.regress(y, [t25, t50, sr])[:2])
    single = np.array(single)                       # (n, 5 assignments, 2 regressors, 14)
    two["c"] = single[:, [1, 2], 0, :]
    return single, {k: np.array(v) for k, v in two.items()}

REG = {}
for g in ("A", "B"):
    for gk in ("low", "int", "exc"):
        for o2, o3 in (READINGS if g == "A" else [("a", "drop"), ("code", "one")]):
            REG[(g, gk, o2, o3)] = regression(GROUPS[g][gk], o2, o3)
print("regressions run:", len(REG), "; neurons per group (A):", {k: len(GROUPS['A'][k]) for k in ('low', 'int', 'exc')})
print("trials with a 0/0 suppression ratio among these neurons:",
      sum(np.isnan(u["sr"]).sum() for k in ("low", "int", "exc") for u in GROUPS["A"][k]))

def rt_vs_fo(single):
    m = single[:, :, :, CUE_BINS].mean(3).mean(0)         # (5, 2): mean cue beta, RT and FO
    return m

def assign_anova(single, a1, a2, bins=slice(None)):
    Y = single[:, [a1, a2], :, :][:, :, :, bins]          # (n, 2, 2, bins)
    return vp.rm_anova(Y, ["assignment", "regressor", "interval"])

### Claim fig4a-low-rt-over-fo

> Across all five uncertainty assignments, Low firing neurons signaled relative threat more strongly
> than fear output (cue-averaged relative-threat betas further from zero than fear-output betas).

In [ ]:
def rt_fo_table(gk, groups=("A", "B")):
    rows = []
    for key, (single, _) in REG.items():
        if key[1] != gk or key[0] not in groups:
            continue
        m = rt_vs_fo(single)
        rows.append({"groups": key[0], "O2": key[2], "O3": key[3],
                     **{f"RT {a}": m[i, 0] for i, a in enumerate(vp.ASSIGNMENTS)},
                     **{f"FO {a}": m[i, 1] for i, a in enumerate(vp.ASSIGNMENTS)},
                     "RT further from 0 at every assignment": bool(np.all(abs(m[:, 0]) > abs(m[:, 1])))})
    return pd.DataFrame(rows)

print(rt_fo_table("low").round(3).to_string(index=False))

**Result: reproduced.** Under every reading the mean cue relative-threat beta of Low firing neurons is further from zero than the fear-output beta at all five assignments (O2 (a), 0/0 dropped: -0.29 to -0.48 against -0.12 to -0.24). Both are negative.

### Claim fig4a-low-025-better-than-000

> For Low firing neurons, relative threat signaling was better captured by the actual shock
> probability (0.25) than by equating uncertainty with safety (0.00); ANOVA on betas gave an
> assignment x regressor x interval interaction.

In [ ]:
def assign_table(gk, a1, a2, groups=("A", "B")):
    rows = []
    for key, (single, _) in REG.items():
        if key[1] != gk or key[0] not in groups:
            continue
        r = assign_anova(single, a1, a2)
        m = rt_vs_fo(single)
        rows.append({"groups": key[0], "O2": key[2], "O3": key[3], "n": single.shape[0],
                     "assignment x regressor x interval": f"F({r['assignment x regressor x interval'][1]},{r['assignment x regressor x interval'][2]}) = {r['assignment x regressor x interval'][0]:.2f}",
                     "p": r["assignment x regressor x interval"][3],
                     f"|RT| at {vp.ASSIGNMENTS[a1]}": abs(m[a1, 0]), f"|RT| at {vp.ASSIGNMENTS[a2]}": abs(m[a2, 0])})
    return pd.DataFrame(rows)

t = assign_table("low", 0, 1)
t["criterion met"] = (t.p < 0.05) & (t["|RT| at 0.25"] > t["|RT| at 0.0"])
print(t.round(4).to_string(index=False))

**Result: reproduced.** The assignment x regressor x interval interaction for 0.00 against 0.25 is significant under every reading (O2 (a), 0/0 dropped, F(13,949) = 2.10, p = 0.012; paper F(13,936) = 2.13, p = 0.011) and the 0.25 relative-threat beta is further from zero (0.40 against 0.29).

### Claim fig4a-low-025-similar-050

> For Low firing neurons, relative threat signaling of the actual probability (0.25) was similar to
> that of the midpoint probability (0.50); ANOVA found no assignment x regressor x interval
> interaction.

In [ ]:
t = assign_table("low", 1, 2)
t["absence (p >= 0.05)"] = t.p >= 0.05
print(t.round(4).to_string(index=False))

**Result: reproduced.** The interaction for 0.25 against 0.50 is not significant under any reading (p = 0.36 to 0.59; paper p = 0.57).

### Claim fig4b-low-dynamic-shift

> Low firing neurons first decreased firing according to the actual shock probability (0.25) and
> later according to the midpoint probability (0.50); ANOVA on betas of the two relative-threat
> regressors gave an assignment x interval interaction.

In [ ]:
def two_table(gk, groups=("A", "B")):
    rows = []
    for key, (_, two) in REG.items():
        if key[1] != gk or key[0] not in groups:
            continue
        for model, B in two.items():
            r = vp.rm_anova(B, ["assignment", "interval"])["assignment x interval"]
            rows.append({"groups": key[0], "O2": key[2], "O3": key[3], "model": model, "n": B.shape[0],
                         "assignment x interval": f"F({r[1]},{r[2]}) = {r[0]:.2f}", "p": r[3]})
    return pd.DataFrame(rows)

t = two_table("low")
t["criterion met (p < 0.05)"] = t.p < 0.05
print(t.round(4).to_string(index=False))

**Result: reproduced under the two-regressor model (a).** With a constant and the 0.25 and 0.50 regressors, the model the authors' code fits, the assignment x interval interaction is significant under every reading (F(13,949) = 2.34 to 2.44, p = 0.003 to 0.005; paper F(13,949) = 2.42, p = 0.003). Adding fear output (model b) gives p = 0.10 to 0.12, and taking the betas from the single-assignment models (c) gives p = 0.30 to 0.48.

### Claims fig4c-low-early-late-betas and fig4d-low-early-late-shift

> In Low firing neurons, early betas (first 4 s of cue) were below zero for the 0.25 assignment but
> not for 0.50, and late betas (last 2 s of cue plus 2 s delay) were below zero for 0.50 but not for
> 0.25.

> In Low firing neurons the early-to-late change in beta was positive for the 0.25 assignment and
> negative for 0.50, and the two shifts differed.

In [ ]:
def early_late(gk, seed, groups=("A",)):
    rng = seed_for(seed)
    out = {}
    for key, (_, two) in REG.items():
        if key[1] != gk or key[0] not in groups:
            continue
        for model, B in two.items():
            e25, e50 = B[:, 0, EARLY].mean(1), B[:, 1, EARLY].mean(1)
            l25, l50 = B[:, 0, LATE_B].mean(1), B[:, 1, LATE_B].mean(1)
            out[key + (model,)] = {"early 0.25": vp.boot_ci(e25, rng), "early 0.50": vp.boot_ci(e50, rng),
                                   "late 0.25": vp.boot_ci(l25, rng), "late 0.50": vp.boot_ci(l50, rng),
                                   "shift 0.25": vp.boot_ci(l25 - e25, rng), "shift 0.50": vp.boot_ci(l50 - e50, rng),
                                   "shift 0.25 - shift 0.50": vp.boot_ci((l25 - e25) - (l50 - e50), rng)}
    return out

def el_table(res, keys, rules):
    rows = []
    for k, v in res.items():
        pat, val = judge(v, rules)
        rows.append({"groups": k[0], "O2": k[2], "O3": k[3], "model": k[4],
                     **{kk: f"{v[kk][0]:.2f} [{v[kk][1]:.2f}, {v[kk][2]:.2f}]" for kk in keys},
                     "pattern holds": pat, "values within tolerance": val})
    return pd.DataFrame(rows)

EL_LOW = early_late("low", "fig4c-low-early-late-betas")
rules_4c = [("early 0.25", "below", -0.57, 0.47), ("early 0.50", "contains", 0.06, 0.44),
            ("late 0.50", "below", -0.51, 0.52), ("late 0.25", "contains", 0.20, 0.42)]
print(el_table(EL_LOW, ["early 0.25", "early 0.50", "late 0.25", "late 0.50"], rules_4c).to_string(index=False))

**Result: partially reproduced.** Under model (a) and every reading of O2 and O3, the early 0.25 interval lies below zero, the early 0.50 interval contains zero and the late 0.50 interval lies below zero, but the late 0.25 interval lies above zero (O2 (a): 0.57 [0.12, 1.00]; paper 0.20 [-0.22, 0.62]). With the authors' per-trial firing all four parts hold and the means are the paper's (-0.57, 0.05, 0.20, -0.51; paper -0.57, 0.06, 0.20, -0.51).

In [ ]:
rules_4d = [("shift 0.25", "above", 0.76, 0.35), ("shift 0.50", "below", -0.56, 0.40),
            ("shift 0.25 - shift 0.50", "above", 1.32, 0.75)]
print(el_table(EL_LOW, ["shift 0.25", "shift 0.50", "shift 0.25 - shift 0.50"], rules_4d).to_string(index=False))

**Result: reproduced.** Under model (a) and every reading, the 0.25 shift is positive, the 0.50 shift negative and their difference positive, with means within tolerance (O2 (a), 0/0 dropped: 1.10, -0.84, 1.94; paper 0.76, -0.56, 1.32). The authors' per-trial firing gives 0.77, -0.57, 1.33.

### Claim fig4e-int-025-better-than-000

> Intermediate firing neurons signaled relative threat more strongly than fear output, and the
> actual probability (0.25) did better than 0.00; ANOVA on betas for 0.00 and 0.25 gave an
> assignment x regressor x interval interaction.

In [ ]:
a = rt_fo_table("int")
t = assign_table("int", 0, 1)
t["RT further from 0 at every assignment"] = a["RT further from 0 at every assignment"].values
t["criterion met"] = t["RT further from 0 at every assignment"] & (t.p < 0.05) & (t["|RT| at 0.25"] > t["|RT| at 0.0"])
print(a.round(3).to_string(index=False))
print(t.round(4).to_string(index=False))

**Result: reproduced.** Under O2 (a) and (c) the relative-threat beta is further from zero than fear output at every assignment, the 0.00 against 0.25 interaction is significant (O2 (a), 0/0 dropped, F(13,429) = 2.12, p = 0.012; paper F(13,416) = 2.10, p = 0.013) and the 0.25 beta is further from zero (0.78 against 0.54). Under O2 (b) fear output is further from zero than relative threat at 1.00.

### Claim fig4e-int-025-similar-050

> For Intermediate firing neurons, ANOVA for 0.25 and 0.50 betas found no assignment x regressor x
> interval interaction.

In [ ]:
t = assign_table("int", 1, 2)
t["absence (p >= 0.05)"] = t.p >= 0.05
print(t.round(4).to_string(index=False))

**Result: reproduced.** The 0.25 against 0.50 interaction is not significant under any reading (p = 0.42 to 0.48; paper p = 0.37).

### Claim fig4f-int-no-assignment-interval

> In Intermediate firing neurons, the ANOVA on the 0.25 and 0.50 relative-threat betas found no
> assignment x interval interaction.

In [ ]:
t = two_table("int")
t["absence (p >= 0.05)"] = t.p >= 0.05
print(t.round(4).to_string(index=False))

**Result: reproduced.** On the paper's Intermediate neurons the assignment x interval interaction is not significant under any reading or model (model a, p = 0.25 to 0.27; paper p = 0.25). On the 54 Intermediate neurons of groups B it is significant under models (a) and (b).

### Claims fig4g-int-early-late-betas and fig4h-int-early-late-shift

> In Intermediate firing neurons early betas were below zero for the 0.25 assignment but not for
> 0.50, and late betas did not differ from zero for either.

> In Intermediate firing neurons the early-to-late shift was positive for 0.25, not different from
> zero for 0.50, and the two shifts differed.

In [ ]:
EL_INT = early_late("int", "fig4g-int-early-late-betas")
rules_4g = [("early 0.25", "below", -0.77, 0.73), ("early 0.50", "contains", 0.07, 1.07),
            ("late 0.25", "contains", 0.13, 0.66), ("late 0.50", "contains", -0.43, 0.62)]
print(el_table(EL_INT, ["early 0.25", "early 0.50", "late 0.25", "late 0.50"], rules_4g).to_string(index=False))

**Result: not reproduced.** No reading with the Methods' normalization gives the pattern. Under model (a) the early 0.25 interval contains zero (O2 (a): -0.40 [-1.08, 0.34]) and the late 0.50 interval lies below zero; under model (c) both early intervals lie below zero. With the authors' per-trial firing the pattern holds and the means are the paper's (-0.76, 0.07, 0.12, -0.44; paper -0.77, 0.07, 0.13, -0.43).

In [ ]:
rules_4h = [("shift 0.25", "above", 0.88, 0.56), ("shift 0.50", "contains", -0.52, 0.60),
            ("shift 0.25 - shift 0.50", "above", 1.38, 1.15)]
print(el_table(EL_INT, ["shift 0.25", "shift 0.50", "shift 0.25 - shift 0.50"], rules_4h).to_string(index=False))

**Result: reproduced.** Under model (a) with O2 (a), (b) or (c) the 0.25 shift lies above zero, the 0.50 shift interval contains zero and the difference of shifts lies above zero, with means within tolerance (O2 (a): 1.39, -0.81, 2.20; paper 0.88, -0.52, 1.38).

### Claim fig4-low-int-combined

> Pooling Low and Intermediate firing neurons, ANOVA found an assignment x interval interaction and
> no neuron type x assignment x interval interaction, so the Intermediate shift did not differ from
> the Low one.

In [ ]:
rows = []
for o2, o3 in READINGS:
    for model in ("a", "b", "c"):
        B = np.concatenate([REG[("A", "low", o2, o3)][1][model], REG[("A", "int", o2, o3)][1][model]])
        grp = np.array([0] * len(GROUPS["A"]["low"]) + [1] * len(GROUPS["A"]["int"]))
        r = vp.rm_anova(B, ["assignment", "interval"], between=grp, between_name="type")
        ai, tai = r["assignment x interval"], r["type x assignment x interval"]
        rows.append({"O2": o2, "O3": o3, "model": model, "assignment x interval": f"F({ai[1]},{ai[2]}) = {ai[0]:.2f}, p = {ai[3]:.3g}",
                     "type x assignment x interval": f"F({tai[1]},{tai[2]}) = {tai[0]:.2f}, p = {tai[3]:.3g}",
                     "criterion met": ai[3] < 0.05 and tai[3] >= 0.05})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Under models (a) and (b) and every reading, assignment x interval is significant (model a, F(13,1378) = 2.51, p = 0.002; paper F(13,1378) = 2.52, p = 0.002) and neuron type x assignment x interval is not (p = 0.70 to 0.75; paper p = 0.74).

### Claim fig6-excited-assignment-050

> In cue-excited neurons, single-unit activity reflected a mixture of relative threat and fear
> output, and relative threat was better captured by the midpoint assignment (0.50) than by 0.25 or
> 0.75, with assignment x regressor interactions in both comparisons.

ANOVA over the 12 bins from cue onset to 2 s after offset (indices 2-13).

In [ ]:
rows = []
for key, (single, _) in REG.items():
    if key[1] != "exc":
        continue
    r1 = assign_anova(single, 1, 2, slice(2, 14))["assignment x regressor"]
    r2 = assign_anova(single, 2, 3, slice(2, 14))["assignment x regressor"]
    m = single[:, :, 0, 2:14].mean((0, 2))
    rows.append({"groups": key[0], "O2": key[2], "O3": key[3], "n": single.shape[0],
                 "0.25 vs 0.50": f"F({r1[1]},{r1[2]}) = {r1[0]:.2f}, p = {r1[3]:.3g}",
                 "0.50 vs 0.75": f"F({r2[1]},{r2[2]}) = {r2[0]:.2f}, p = {r2[3]:.3g}",
                 "mean RT beta 0.25/0.50/0.75": "/".join(f"{x:.3f}" for x in m[1:4]),
                 "criterion met": r1[3] < 0.05 and r2[3] <= 0.05 and m[2] > max(m[1], m[3])})
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** Under every reading with the Methods' normalization the 0.25 against 0.50 assignment x regressor interaction is significant (p = 0.002 to 0.022) and the mean relative-threat beta is largest at 0.50, but the 0.50 against 0.75 interaction is not (p = 0.12 to 0.30; paper p = 0.05). With the authors' per-trial firing it is (F(1,130) = 3.93, p = 0.0497; paper F(1,129) = 3.92).

### Claim fig6c-excited-regressor-interval

> In cue-excited neurons, early signaling of elevated relative threat gave way to joint signaling of
> relative threat and fear output until shock; ANOVA on the 0.50 and fear output betas gave a main
> effect of regressor and a regressor x interval interaction.

In [ ]:
rows = []
for key, (single, _) in REG.items():
    if key[1] != "exc":
        continue
    r = vp.rm_anova(single[:, 2, :, :], ["regressor", "interval"])
    rows.append({"groups": key[0], "O2": key[2], "O3": key[3], "n": single.shape[0],
                 "regressor": f"F({r['regressor'][1]},{r['regressor'][2]}) = {r['regressor'][0]:.2f}, p = {r['regressor'][3]:.3g}",
                 "regressor x interval": f"F({r['regressor x interval'][1]},{r['regressor x interval'][2]}) = {r['regressor x interval'][0]:.2f}, p = {r['regressor x interval'][3]:.3g}",
                 "criterion met": r["regressor"][3] < 0.05 and r["regressor x interval"][3] < 0.05})
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** No reading with the Methods' normalization gives both effects: O2 (b) gives the regressor main effect (p = 0.002 to 0.003) without the interaction (p = 0.058 to 0.068), O2 (c) the interaction (p = 0.021 to 0.027) without the main effect (p = 0.054 to 0.056), and O2 (a) neither. The authors' per-trial firing gives both (regressor p = 0.011, regressor x interval p = 0.013; paper p = 0.013 and 0.01).

### Claim fig6d-excited-betas

> In cue-excited neurons, relative-threat (0.50) betas exceeded zero at onset, late cue and delay,
> fear-output betas exceeded zero in late cue and delay, and the two differed at onset but not in
> late cue or delay.

Periods on the 1 s bins: onset 0-1 s (index 2), late cue 5-10 s (7-11), delay 10-12 s (12-13).

In [ ]:
rng = seed_for("fig6d-excited-betas")
REG_PERIODS = {"onset": [2], "late cue": list(range(7, 12)), "delay": [12, 13]}
rules_6d = [(("RT", "onset"), "above", 0.59, 0.18), (("RT", "late cue"), "above", 0.38, 0.17),
            (("FO", "late cue"), "above", 0.19, 0.11), (("RT", "delay"), "above", 0.43, 0.18),
            (("FO", "delay"), "above", 0.17, 0.12), (("FO", "onset"), "contains", None, 0),
            (("RT-FO", "onset"), "above", 0.48, 0.28), (("RT-FO", "late cue"), "contains", 0.19, 0.25),
            (("RT-FO", "delay"), "contains", 0.27, 0.29)]
rows = []
for key, (single, _) in REG.items():
    if key[1] != "exc":
        continue
    res = {}
    for p, idx in REG_PERIODS.items():
        rt, fo = single[:, 2, 0, idx].mean(1), single[:, 2, 1, idx].mean(1)
        res[("RT", p)], res[("FO", p)], res[("RT-FO", p)] = vp.boot_ci(rt, rng), vp.boot_ci(fo, rng), vp.boot_ci(rt - fo, rng)
    pat, val = judge(res, rules_6d)
    rows.append({"groups": key[0], "O2": key[2], "O3": key[3],
                 **{f"{a} {b}": f"{v[0]:.2f} [{v[1]:.2f}, {v[2]:.2f}]" for (a, b), v in res.items()},
                 "pattern holds": pat, "values within tolerance": val})
print(pd.DataFrame(rows).T.to_string())

**Result: reproduced.** Under O2 (a) and (c) the five beta intervals lie above zero, the fear-output onset interval contains zero, relative threat and fear output differ at onset but not in late cue or delay, and the means are within tolerance (O2 (a), 0/0 dropped: relative threat 0.67, 0.42, 0.49; fear output 0.27, 0.25 in late cue and delay; paper 0.59, 0.38, 0.43, 0.19, 0.17). Under O2 (b) the late-cue and delay differences exclude zero.

## Reward firing (claims fig5*, fig6ef, fig6g, fig6-excited-peak-cue-vs-reward)

Pellet deliveries outside cues and within recorded time, firing in 16 bins of 250 ms from 2 s
before to 2 s after, as described above for pauses. Readings O4: events (a) outside cue and delay,
(b) outside the whole trial window; scale (i) or (ii); the authors' code as a variant. Pre = the
500 ms before delivery (bins 6-7), post = the first 500 ms after (bins 8-9). Correlation values
(O5): reward = (a) the 250 ms bin after delivery minus the bin before, or (b) the larger of the two
minus the bin before; cue = the second 250 ms cue bin (cue-inhibited neurons) or the first
(cue-excited) minus the last bin before cue onset (`vpAnalysis.m`, lines 2778-2779 and 3903-3904,
use these bins).

In [ ]:
REWARD_READINGS = [("a", "i"), ("a", "ii"), ("b", "i"), ("b", "ii"), ("code", "i"), ("code", "ii"), ("code", "code")]
RW = {}
for gk in ("low", "int", "exc"):
    for reading, scale in REWARD_READINGS:
        M, kept = event_matrix(GROUPS["A"][gk], "reward", reading, scale)
        RW[(gk, reading, scale)] = (M, kept)
ev_counts = {r: np.median([len(vp.reward_events(u, r)) for u in GROUPS["A"]["low"] + GROUPS["A"]["int"] + GROUPS["A"]["exc"]])
             for r in ("a", "b", "code")}
print("median pellets per neuron by event reading:", ev_counts)
print(pd.DataFrame([{"group": k[0], "events": k[1], "scale": k[2], "neurons kept": f"{len(v[1])} of {len(GROUPS['A'][k[0]])}",
                     "range of normalized firing": f"{v[0].min():.2f} to {v[0].max():.2f}"} for k, v in RW.items()]).to_string(index=False))

### Claim fig5a-low-reward-firing

> Low firing neurons sharply increased firing after reward delivery and Intermediate firing neurons
> did not; ANOVA gave a cluster x interval interaction and a main effect of interval in Low but not
> Intermediate neurons.

In [ ]:
rows = []
for reading, scale in REWARD_READINGS:
    Ml, Mi = RW[("low", reading, scale)][0], RW[("int", reading, scale)][0]
    r = vp.rm_anova(np.concatenate([Ml, Mi]), ["interval"], between=np.r_[np.zeros(len(Ml)), np.ones(len(Mi))],
                    between_name="cluster")["cluster x interval"]
    rl, ri = vp.rm_anova(Ml, ["interval"])["interval"], vp.rm_anova(Mi, ["interval"])["interval"]
    rows.append({"events": reading, "scale": scale, "cluster x interval": f"F({r[1]},{r[2]}) = {r[0]:.2f}, p = {r[3]:.3g}",
                 "Low interval": f"F({rl[1]},{rl[2]}) = {rl[0]:.2f}, p = {rl[3]:.3g}",
                 "Int interval": f"F({ri[1]},{ri[2]}) = {ri[0]:.2f}, p = {ri[3]:.3g}",
                 "criterion met": r[3] < 0.05 and rl[3] < 0.05 and ri[3] >= 0.05})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Under every reading cluster x interval is significant (events (a), scaling (i): F(15,1590) = 3.68, p = 2.2e-6), Low neurons show an interval effect (p = 1.4e-9) and Intermediate neurons do not (p = 0.11; paper F(15,495) = 1.21, p = 0.26).

### Claim fig5bc-reward-pre-post

> In Low firing neurons pre-reward firing hovered around zero, while post-reward firing exceeded
> pre-reward firing and zero; in Intermediate firing neurons neither pre- nor post-reward firing
> differed from zero.

In [ ]:
rng = seed_for("fig5bc-reward-pre-post")
rows = []
for reading, scale in REWARD_READINGS:
    res = {}
    for gk in ("low", "int"):
        M = RW[(gk, reading, scale)][0]
        pre, post = M[:, 6:8].mean(1), M[:, 8:10].mean(1)
        res[(gk, "pre")], res[(gk, "post")], res[(gk, "post-pre")] = vp.boot_ci(pre, rng), vp.boot_ci(post, rng), vp.boot_ci(post - pre, rng)
    pat, val = judge(res, [(("low", "pre"), "contains", -0.06, 0.09), (("low", "post-pre"), "above", 0.34, 0.19),
                           (("low", "post"), "above", 0.28, 0.14), (("int", "pre"), "contains", 0.00026, 0.17),
                           (("int", "post"), "contains", -0.14, 0.27)])
    rows.append({"events": reading, "scale": scale, **{f"{a} {b}": f"{v[0]:.3f} [{v[1]:.3f}, {v[2]:.3f}]" for (a, b), v in res.items()},
                 "pattern holds": pat, "values within tolerance": val})
print(pd.DataFrame(rows).T.to_string())

**Result: reproduced.** With scaling (i) and either event reading, the Low post and post - pre intervals lie above zero, the Low pre interval and both Intermediate intervals contain zero, and the means are within tolerance (events a: Low pre -0.078, post 0.368, post - pre 0.446; Intermediate pre 0.012, post -0.257). With scaling (ii) the Low post means are about three times larger, the Intermediate pre interval excludes zero under both event readings, and the Low pre interval excludes zero only under events (b) ([0.001, 0.243]; events a [-0.044, 0.150]).

### Claims fig5d-f-low-reward-cue-correlations, fig5-low-fisher-comparisons and fig5g-i-intermediate-no-reward-cue-correlation

> Across Low firing neurons, larger reward firing increases went with larger firing decreases to
> danger and to uncertainty (negative correlations), with no relationship between reward and safety
> firing.

> In Low firing neurons the danger-reward and uncertainty-reward correlations did not differ, while
> the uncertainty-reward and safety-reward correlations did.

> Intermediate firing neurons showed no cue-reward firing relationships, and their correlations did
> not differ from one another.

Fisher comparisons: independent-samples form (the claims file's main reading), with Steiger's test
for dependent correlations as a variant.

In [ ]:
def reward_value(M, o5):
    return M[:, 8] - M[:, 7] if o5 == "a" else np.maximum(M[:, 7], M[:, 8]) - M[:, 7]

def cue_values(us, pool, first_bin):
    Z = cue_array(us, pool)
    return {c: Z[:, i, first_bin] - Z[:, i, 79] for i, c in enumerate("DUS")}

def corr_rows(gk, cue_bin):
    rows = []
    for pool in ("3", "4", "code"):
        for reading, scale in REWARD_READINGS:
            if (pool == "code") != (scale == "code"):
                continue
            M, kept = RW[(gk, reading, scale)]
            cv = cue_values(kept, pool, cue_bin)
            for o5 in ("a", "b"):
                rv = reward_value(M, o5)
                rs = {c: stats.pearsonr(rv, cv[c]) for c in "DUS"}
                cvd = {c: cv[c] for c in "DUS"}
                rows.append({"pool": pool, "events": reading, "scale": scale, "O5": o5, "n": len(kept),
                             **{f"r {c}": rs[c].statistic for c in "DUS"}, **{f"R2 {c}": rs[c].statistic ** 2 for c in "DUS"},
                             **{f"p {c}": rs[c].pvalue for c in "DUS"},
                             "_rv": rv, "_cv": cvd})
    return pd.DataFrame(rows)

c5 = corr_rows("low", 81)
c5["pattern holds"] = (c5["r D"] < 0) & (c5["p D"] < 0.05) & (c5["r U"] < 0) & (c5["p U"] < 0.05) & (c5["p S"] >= 0.05)
c5["R2 within tolerance"] = (abs(c5["R2 D"] - 0.08) <= 0.008) & (abs(c5["R2 U"] - 0.16) <= 0.016) & (abs(c5["R2 S"] - 0.006) <= 0.0006)
print(c5.drop(columns=["_rv", "_cv"]).round(4).to_string(index=False))

**Result: partially reproduced.** Under every reading reward firing correlates negatively with danger and uncertainty firing (p < 0.05) and not with safety firing, but no reading with the Methods' normalization puts all three R2 within the claims file's 10% tolerance (events a, scaling i, O5 a: R2 = 0.067, 0.154, 0.005; paper 0.08, 0.16, 0.006). With the authors' normalization and O5 (a) all three are within tolerance (0.084, 0.156, 0.0057). The partial outcome comes from the 10% tolerance on R2, which is asked of a reader (questions/crcns-bf-3.yaml).


In [ ]:
rows = []
for _, r in c5.iterrows():
    z1, p1 = vp.fisher_z(r["r D"], r["r U"], r["n"], r["n"])
    z2, p2 = vp.fisher_z(r["r U"], r["r S"], r["n"], r["n"])
    s1 = vp.steiger_z(r["r D"], r["r U"], stats.pearsonr(r["_cv"]["D"], r["_cv"]["U"]).statistic, r["n"])
    s2 = vp.steiger_z(r["r U"], r["r S"], stats.pearsonr(r["_cv"]["U"], r["_cv"]["S"]).statistic, r["n"])
    rows.append({"pool": r["pool"], "events": r["events"], "scale": r["scale"], "O5": r["O5"],
                 "Z D vs U": z1, "p D vs U": p1, "Z U vs S": z2, "p U vs S": p2,
                 "Steiger Z D vs U": s1[0], "Steiger p": s1[1], "Steiger Z U vs S": s2[0], "Steiger p ": s2[1],
                 "criterion met": p1 >= 0.05 and p2 < 0.05})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("paper: Z = 0.74 (p = 0.46) and Z = 2.96 (p = 0.0031); the comparisons use correlations whose signs are those printed above")

**Result: reproduced.** Under every reading the danger-reward and uncertainty-reward correlations do not differ (independent-samples Z = 0.70 to 1.39, p >= 0.17) and the uncertainty-reward and safety-reward correlations do (|Z| = 2.77 to 3.35, p <= 0.006); the authors' normalization gives Z = 0.71 and 2.94 (paper 0.74 and 2.96). Steiger's test for dependent correlations gives the same outcome.

In [ ]:
c5i = corr_rows("int", 81)
rows = []
for _, r in c5i.iterrows():
    zs = [vp.fisher_z(r[f"r {a}"], r[f"r {b}"], r["n"], r["n"]) for a, b in (("D", "U"), ("U", "S"), ("D", "S"))]
    rows.append({"pool": r["pool"], "events": r["events"], "scale": r["scale"], "O5": r["O5"], "n": r["n"],
                 **{f"r {c}": r[f"r {c}"] for c in "DUS"}, **{f"p {c}": r[f"p {c}"] for c in "DUS"},
                 "Z D-U / U-S / D-S": "/".join(f"{z:.2f}" for z, _ in zs), "min Fisher p": min(p for _, p in zs),
                 "absence (all p >= 0.05)": all(r[f"p {c}"] >= 0.05 for c in "DUS") and all(p >= 0.05 for _, p in zs)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** For Intermediate neurons none of the three cue-reward correlations is significant (p >= 0.14) and no Fisher comparison is (p >= 0.31), under every reading.

### Claim fig6ef-excited-reward-firing

> Cue-excited neurons increased firing to reward; ANOVA gave a main effect of interval, and
> post-reward firing exceeded zero and pre-reward firing.

In [ ]:
rng = seed_for("fig6ef-excited-reward-firing")
rows = []
for reading, scale in REWARD_READINGS:
    M = RW[("exc", reading, scale)][0]
    r = vp.rm_anova(M, ["interval"])["interval"]
    pre, post = M[:, 6:8].mean(1), M[:, 8:10].mean(1)
    res = {"post": vp.boot_ci(post, rng), "post-pre": vp.boot_ci(post - pre, rng)}
    pat, val = judge(res, [("post", "above", 0.22, 0.15), ("post-pre", "above", 0.26, 0.16)])
    rows.append({"events": reading, "scale": scale, "n": len(M), "interval": f"F({r[1]},{r[2]}) = {r[0]:.2f}, p = {r[3]:.3g}",
                 **{k: ci_text(*v) for k, v in res.items()}, "pattern holds": pat and r[3] < 0.05, "values within tolerance": val})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** With scaling (i) and either event reading, the interval effect is significant (events a: F(15,1950) = 3.84, p = 8e-7) and post-reward firing exceeds zero and pre-reward firing, within tolerance (0.239 and 0.312; paper 0.22 and 0.26). With scaling (ii) the means are about four times larger.

### Claim fig6g-excited-reward-cue-correlations

> Across cue-excited neurons, larger firing increases at reward peak went with larger increases at
> danger, uncertainty and safety peaks.

In [ ]:
c6 = corr_rows("exc", 80)
c6["pattern holds"] = (c6["r D"] > 0) & (c6["p D"] < 0.05) & (c6["r U"] > 0) & (c6["p U"] < 0.05) & (c6["r S"] > 0) & (c6["p S"] < 0.05)
c6["R2 within tolerance"] = (abs(c6["R2 D"] - 0.09) <= 0.009) & (abs(c6["R2 U"] - 0.12) <= 0.012) & (abs(c6["R2 S"] - 0.08) <= 0.008)
print(c6.drop(columns=["_rv", "_cv"]).round(4).to_string(index=False))

**Result: partially reproduced.** Under every reading reward firing correlates positively with danger, uncertainty and safety firing (p < 0.05), but no reading puts all three R2 within the 10% tolerance (events a, scaling i, O5 a: 0.101, 0.100, 0.076; paper 0.09, 0.12, 0.08); the authors' normalization gives 0.084, 0.113, 0.098. The partial outcome comes from the 10% tolerance on R2, which is asked of a reader (questions/crcns-bf-3.yaml).


### Claim fig6-excited-peak-cue-vs-reward

> In cue-excited neurons peak danger firing differed from peak reward firing, while peak uncertainty
> and safety firing did not.

In [ ]:
rng = seed_for("fig6-excited-peak-cue-vs-reward")
rows = []
for _, r in c6.iterrows():
    res = {c: vp.boot_ci(r["_cv"][c] - r["_rv"], rng) for c in "DUS"}
    pat, val = judge(res, [("D", "above", 0.40, 0.27), ("U", "contains", 0.13, 0.22), ("S", "contains", -0.08, 0.22)])
    rows.append({"pool": r["pool"], "events": r["events"], "scale": r["scale"], "O5": r["O5"],
                 **{f"{c} - reward": ci_text(*v) for c, v in res.items()}, "pattern holds": pat, "values within tolerance": val})
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** With the Methods' normalization only the three-cue pool with scaling (i) and O5 (b) gives the pattern (danger - reward above zero, the other two containing zero), with means outside tolerance (events a: 0.632, 0.089, -0.368; paper 0.40, 0.13, -0.08); with O5 (a) and scaling (i) uncertainty - reward also lies above zero, and with scaling (ii) the pattern fails. With the authors' normalization and O5 (a) the pattern holds and the means are the paper's (0.398, 0.136, -0.086). Under scaling (i) the cue values are Z-scored over the cue bins and the reward values over the 16 reward bins, so their difference has no common unit; under scaling (ii), where both share the cue normalization's SD, no reading gives the pattern. Whether scaling (i) belongs among this claim's readings is asked of a reader (questions/crcns-bf-3.yaml).


## Results by rat

The neurons of each group come from several rats. Per rat with at least one neuron in the group: the
number of neurons and the mean over them of danger firing at onset and in late cue (three-cue pool),
of the reward post - pre difference (events a, scaling i), and of the early and late 0.25 betas and
the late 0.50 beta of the two-regressor model (model a, O2 a, 0/0 dropped).

In [ ]:
by_rat = []
for gk in ("low", "int", "exc"):
    us = GROUPS["A"][gk]
    Z = cue_array(us, "3")
    M, kept = RW[(gk, "a", "i")]
    rw = dict(zip([u["name"] for u in kept], M[:, 8:10].mean(1) - M[:, 6:8].mean(1)))
    two = REG[("A", gk, "a", "drop")][1]["a"]
    for rat in sorted({u["rat"] for u in us}):
        idx = [i for i, u in enumerate(us) if u["rat"] == rat]
        by_rat.append({"group": gk, "rat": rat, "neurons": len(idx),
                       "danger onset": Z[idx, 0][:, vp.ONSET].mean(), "danger late cue": Z[idx, 0][:, vp.LATE].mean(),
                       "reward post - pre": np.mean([rw[us[i]["name"]] for i in idx if us[i]["name"] in rw]),
                       "early 0.25 beta": two[idx, 0][:, EARLY].mean(), "late 0.25 beta": two[idx, 0][:, LATE_B].mean(),
                       "late 0.50 beta": two[idx, 1][:, LATE_B].mean()})
by_rat = pd.DataFrame(by_rat)
print(by_rat.round(3).to_string(index=False))
for gk, sign in (("low", -1), ("int", -1), ("exc", 1)):
    d = by_rat[by_rat.group == gk]
    print(f"{gk}: rats {len(d)}; danger onset firing with the group's sign in {(np.sign(d['danger onset']) == sign).sum()}; "
          f"reward post - pre above zero in {(d['reward post - pre'] > 0).sum()}; "
          f"largest share of the group's neurons from one rat {d.neurons.max() / d.neurons.sum():.2f}")

Danger onset firing has the group's sign in every rat (11 rats with Low, 9 with Intermediate, 14
with cue-excited neurons), so the cue results are not carried by one animal. The reward response
varies more: post - pre is above zero in 8 of 11 rats for Low neurons, 9 of 14 for cue-excited
neurons and 2 of 9 for Intermediate neurons. The regression betas of rats with one or two neurons
range widely. One rat gives 32% of the Intermediate neurons (pa21) and, as claim clusters-by-rat
shows, 11 of 18 High neurons come from pa02.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome | Note |
|---|---|---|
| recorded-units | partially reproduced | 435 neurons and 14 rats match; 195 sessions, not 194 |
| cue-responsive-screen | reproduced | 271 / 160 with the Methods' rates; 257 / 153 with the code's floored rates |
| fig1f-suppression-cue-effect | reproduced |  |
| fig1f-differential-suppression | reproduced |  |
| fig2-excited-inhibited-split | reproduced (A1) | Hz, two means, 130/127 on the 257; not on the Methods' 271 neurons |
| ancova-excited-no-baseline-effect | reproduced (A1) | significant on groups B |
| ancova-inhibited-baseline-effect | reproduced |  |
| kmeans-three-clusters | reproduced (A2) | a local optimum; 100 restarts give other clusters |
| clusters-differ-in-cue-firing | partially reproduced (A2) | Low vs Intermediate p = 0.035, paper p < 0.005 |
| clusters-by-rat | reproduced (A2) |  |
| fig3a-low-differential-cue-firing | reproduced (A2) |  |
| fig3b-low-inhibited-to-threat-cues | partially reproduced (A2) | means twice the paper's; code normalization matches |
| fig3b-low-differential | partially reproduced (A2) | danger - uncertainty means outside tolerance; with code normalization the delay interval reaches zero |
| fig3c-intermediate-differential-cue-firing | reproduced (A2) |  |
| fig3d-intermediate-inhibition | partially reproduced (A2) | as above |
| fig3d-intermediate-differential | partially reproduced (A2) | late-cue danger - uncertainty contains zero |
| supp5-low-cue-correlations | not reproduced (A2) | Low onset uncertainty-safety r = 0.36 |
| supp6-low-nosepoke-pauses | partially reproduced (A2, A3) | Intermediate post-pause firing below zero with the authors' scaling and with 5 s / (ii) |
| fig4a-low-rt-over-fo | reproduced (A2) |  |
| fig4a-low-025-better-than-000 | reproduced (A2) |  |
| fig4a-low-025-similar-050 | reproduced (A2) |  |
| fig4b-low-dynamic-shift | reproduced (A2) | model (a) |
| fig4c-low-early-late-betas | partially reproduced (A2) | late 0.25 beta above zero; code firing matches |
| fig4d-low-early-late-shift | reproduced (A2) |  |
| fig4e-int-025-better-than-000 | reproduced (A2) |  |
| fig4e-int-025-similar-050 | reproduced (A2) |  |
| fig4f-int-no-assignment-interval | reproduced (A2) |  |
| fig4g-int-early-late-betas | not reproduced (A2) | code firing matches |
| fig4h-int-early-late-shift | reproduced (A2) |  |
| fig4-low-int-combined | reproduced (A2) |  |
| fig5a-low-reward-firing | reproduced (A2) |  |
| fig5bc-reward-pre-post | reproduced (A2) |  |
| fig5d-f-low-reward-cue-correlations | partially reproduced (A2) | R2 outside 10%; code normalization matches |
| fig5-low-fisher-comparisons | reproduced (A2) |  |
| fig5g-i-intermediate-no-reward-cue-correlation | reproduced (A2) |  |
| fig6a-excited-differential-cue-firing | reproduced (A1) |  |
| fig6b-excited-period-firing | partially reproduced (A1) | delay safety firing below zero |
| supp8-excited-cue-correlations | partially reproduced (A1) | delay correlation not significant |
| supp9-excited-nosepoke-pauses | reproduced (A1, A3) | rises after 5 s pauses with scaling (ii) |
| fig6-excited-assignment-050 | partially reproduced (A1) | 0.50 vs 0.75 p = 0.12 to 0.30 |
| fig6c-excited-regressor-interval | partially reproduced (A1) | never both effects; code firing gives both |
| fig6d-excited-betas | reproduced (A1) |  |
| fig6ef-excited-reward-firing | reproduced (A1) |  |
| fig6g-excited-reward-cue-correlations | partially reproduced (A1) | R2 outside 10% |
| fig6-excited-peak-cue-vs-reward | partially reproduced (A1) | code normalization matches |

Of 45 claims, 28 are reproduced, 15 partially reproduced and 2 not reproduced (the attempt
record gives the counts). The claims on the cue-inhibited clusters rest on A2, and
fig2-excited-inhibited-split, ancova-excited-no-baseline-effect and the claims on cue-excited
neurons on A1; the two pause claims rest on A3 as well. recorded-units, cue-responsive-screen, the
two Fig. 1f claims and ancova-inhibited-baseline-effect rest on none.

**How the analysis differs from the paper's.** Python in place of Matlab: least squares in place of
`regress`, a studentized bootstrap with analytic standard errors in place of `bootci`, a
univariate repeated measures ANOVA written here. The authors' single k-means start is repeated
with numpy's Mersenne Twister, which gives Matlab's `rng(1)` stream. Firing is normalized as the
Methods describe it (trial-averaged rates minus the 10 s baseline of the trial type, Z-scored over
all bins), which gives values about twice the paper's; the authors' code Z-scores single trials
after subtracting each trial's 2 s baseline, floored, and gives the paper's values. Changing one
element at a time, Z over single trials puts all seven checked means within tolerance and Z over
trial means none, whatever the baseline and floor. Ten of the 15 partial outcomes meet the criterion
under the code's variant: four failed on values alone (fig3b-low-inhibited-to-threat-cues,
fig3d-intermediate-inhibition, fig5d-f-low-reward-cue-correlations,
fig6-excited-peak-cue-vs-reward), the other six on p values or intervals. Five do not meet it under
the code's variant either: recorded-units (195 sessions), fig3b-low-differential (the delay
interval reaches zero), supp6-low-nosepoke-pauses (Intermediate firing below zero after pauses),
fig6b-excited-period-firing (safety intervals below zero) and fig6g-excited-reward-cue-correlations
(R2 outside tolerance). The neuron
groups are the exception to following the Methods (A1): they are the code's, which give the
paper's counts; the Methods' screen gives 271 neurons, and on the 257 the claims-file split
closest to the paper's counts differs from the code's on 25 neurons.

**What was found about the data set.**

- Spikes, pokes and pellets were recorded only from 20 s before each cue onset to 31.5 s after it
  (recording paused at EVT14); the rest of each inter-trial interval is not in the files. The data
  descriptor does not say so. "Inter-trial interval" reward and pause events can only come from
  these windows.
- The descriptor does not name the event channels or the cue channel of each behavior program; only
  the authors' script does. The mapping was checked against the shocks (every danger trial and 2
  of 8 uncertainty trials shocked 12 s after onset, no safety trial). EVT08 repeats EVT01.
- The descriptor says `VPcube.poke.base` is the poke rate over the 2 s before the cue; it is over
  the 20 s before. It lists half duration and amplitude ratio among the cube's fields; the cube
  has neither. It describes one mean waveform per trial type; each unit has one.
- 195 sessions, one more than the paper's 194; each has its own trial table.
- Four pairs of units recorded in one session share 70 to 88% of their spikes and may be one
  neuron on two wires.
- The paper's Methods and the authors' code disagree on the normalization of firing, on rounding
  firing rates in the screen, and on the reward baseline (the code subtracts the first second of
  the 4 s window, the Methods the second before delivery). The paper's values follow the code.
- The paper's k-means clusters are a local optimum of k-means, reached by one start in 98 of 300
  seeds; the lowest-SSE partition found moves 20 of the 126 neurons.

In [ ]:
print(f"notebook run time: {(time.time() - started) / 60:.1f} min")